# Packages

In [45]:
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import plotly
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from datetime import datetime, timedelta, date
import math
import os
from pathlib import Path
import re
import gc
from tqdm import tqdm

import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_pacf

import scipy.stats as stats
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import RobustScaler, LabelEncoder, OrdinalEncoder

pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)
pd.set_option('display.float_format', lambda x: "%.4f" % x)
pd.options.plotting.backend = "plotly"

plt.style.use('ggplot')
sns.set_style('darkgrid')

In [46]:
# helper functions
def get_info(df):
    missing_values_train = pd.DataFrame({'Feature': df.columns,
                              'No. of Missing Values': df.isnull().sum().values,
                              '% of Missing Values': ((df.isnull().sum().values)/len(df)*100)})

    unique_values = pd.DataFrame({'Feature': df.columns,
                                'No. of Unique Values': df.nunique().values})

    feature_types = pd.DataFrame({'Feature': df.columns,
                                'DataType': df.dtypes})

    merged_df = pd.merge(missing_values_train, unique_values, on='Feature', how='left')
    merged_df = pd.merge(merged_df, feature_types, on='Feature', how='left')

    return merged_df

def reduce_memory_usage(df: pd.DataFrame, verbose=False) -> pd.DataFrame:
    start_mem = df.memory_usage(deep=True).sum() / 1024**2
    if verbose:
        print(f"Memory usage before: {start_mem:.2f} MB")
    
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type == 'float64':
            df[col] = pd.to_numeric(df[col], downcast='float')
        elif col_type == 'int64':
            df[col] = pd.to_numeric(df[col], downcast='integer')
    
    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    if verbose:
        print(f"Memory usage after: {end_mem:.2f} MB")
        print(f"Reduced by {(start_mem - end_mem) / start_mem * 100:.1f}%")
    
    return df

# Pulling Data

## New Jersey

In [47]:
nj_path = "./data/NewJersey/nj_data.csv"
nj_df = pd.read_csv(nj_path, encoding='latin-1')
nj_df = reduce_memory_usage(nj_df, verbose=True)
print(nj_df.shape)
nj_df.head()

/var/folders/8r/qgk4_l1j20gfq28wvs_pxh2r0000gn/T/ipykernel_4510/3470814138.py:2: DtypeWarning: Columns (0: Officer In Uniform) have mixed types. Specify dtype option on import or set low_memory=False.
  nj_df = pd.read_csv(nj_path, encoding='latin-1')


Memory usage before: 226.74 MB
Memory usage after: 225.08 MB
Reduced by 0.7%
(96688, 46)


,Form ID,County,Agency Name,Officer Name,User ID,Incident ID,Report Number,Incident Case Number,Incident Date,Other Officer Involved,Officer In Uniform,Incident Municipality,Indoor Or Outdoor,Incident Weather,Video Footage,Video Type,Incident Lighting,Location Type,Incident Type,Contact Origin,Planned Contact,Officer Age,Officer Race/Ethnicity,Officer Rank,Officer Gender,Officer Injury Type,Officer Injuries Injured,Officer Medical Treatment,Officer Hospital Treatment,Total Sub Injured In Incident,Subject Injured In Incident,Subject Injured Prior To Incident,Perceived Condition Of Subject,Subject Actions,Subject Resistance,Subject Medical Treatment,Subject Injury Type,Subject Arrested,Reason Subject Not Arrested,Subject Type,Subject Age,Subject Race/Ethnicity,Subject Gender,Force Type,Incident Year,KEEP/DROP
0,866515,Ocean,Berkeley Twp PD,Michael Bulwinski,10818.0000,OCEAN-BERKELEY TWP PD-23-003647,UOF23-11-52,23-003647,11/6/2023,Not Provided,True,"Berkeley, Ocean County",Outdoors,Clear,Yes,Body Worn,Darkness,Street,Assault,Dispatched,NaN,33,White,Officer,Male,Not injured,FALSE,NaN,NaN,0.0000,No,Unknown,No unusual condition noted,Attempt to escape from Custody,Passive Resistor,NaN,NaN,TRUE,NaN,Person,28,Hispanic,Male,Used take down on,2023,KEEP
1,91681,Ocean,Toms River Twp PD,Frank Moschella,11497.0000,OCEAN-TOMS RIVER TWP PD-21-20078,UOF21-5-48,21-20078,5/13/2021,TRUE,True,"Dover, Ocean County",Indoors,Clear,Yes,Motor Vehicle,Dawn/Dusk,Business,"Welfare Check, Potential Mental Health Incident",Officer Dispatched,NaN,34,Not Provided,Not Provided,Other,Not injured,FALSE,NaN,NaN,0.0000,No,No,Potential Mental Health Incident,"Resisted arrest/police officer control, Verbal...","Verbal, Resistive tension (stiffening, tigheni...",NaN,NaN,FALSE,Medical/Mental Health Incident,Person,55,White,Female,Used arms,2021,KEEP
2,50307,Cumberland,Vineland PD,Derrick Magee,4387.0000,CUMBERLAND-VINELAND PD-2021-4288,UOF21-1-19,2021-4288,1/27/2021,TRUE,True,"Vineland, Cumberland County",Outdoors,Clear,Yes,Body Worn,Artificial,Other,MV/Traffic Stop,Officer Initiated,NaN,30,Not Provided,Not Provided,Other,Not injured,FALSE,NaN,NaN,1.0000,Yes,No,No unusual condition noted,"Resisted arrest/police officer control, Attack...","Resistive tension (stiffening, tighening muscl...","EMS on scene, Hospital",Abrasion/Laceration/Puncture,TRUE,NaN,Person,37,Hispanic,Male,Canine bit (apprehension),2021,KEEP
3,201634,Ocean,Lakewood PD,Deon Smith,11159.0000,OCEAN-LAKEWOOD PD-22-001779,UOF22-1-9,22-001779,1/8/2022,Not Provided,True,"Lakewood, Ocean County",Indoors,NaN,Yes,Body Worn,Artificial,Hospital,Medical Emergency,Dispatched,NaN,28,Not Provided,Not Provided,Other,Not injured,FALSE,NaN,NaN,0.0000,No,No,Other unusual condition noted,"Strike with open hand, fist, or elbow",Active Resistor,NaN,NaN,FALSE,Medical/Mental Health Incident,Person,35,Hispanic,Female,Used arms/hands,2022,KEEP
4,1594362,Bergen,Rutherford PD,Michael Merli,7316.0000,BERGEN-RUTHERFORD PD-25-08463,UOF25-5-1,25-08463,5/11/2025,Not Provided,True,"Rutherford, Bergen County",Indoors,NaN,Yes,"Body Worn, CED Camera",Artificial,Business,Theft/Shoplifting,Dispatched,NaN,47,Not Provided,Not Provided,Other,Not injured,FALSE,NaN,NaN,0.0000,No,No,No unusual condition noted,Attempt to escape from Custody,Active Resistor,NaN,NaN,TRUE,NaN,Person,19,Hispanic,Male,Used take down on,2025,KEEP


In [48]:
nj_df["Incident Case Number"].value_counts()

Incident Case Number
24-29560      24
25-031332     16
2024-02894    15
22-033415     15
20-080948     14
              ..
2025-2758      1
C2043822       1
25-011543      1
P250372236     1
C22040458      1
Name: count, Length: 51743, dtype: int64

In [49]:
nj_df[lambda x: x["Incident Case Number"] == "24-29560"]

,Form ID,County,Agency Name,Officer Name,User ID,Incident ID,Report Number,Incident Case Number,Incident Date,Other Officer Involved,Officer In Uniform,Incident Municipality,Indoor Or Outdoor,Incident Weather,Video Footage,Video Type,Incident Lighting,Location Type,Incident Type,Contact Origin,Planned Contact,Officer Age,Officer Race/Ethnicity,Officer Rank,Officer Gender,Officer Injury Type,Officer Injuries Injured,Officer Medical Treatment,Officer Hospital Treatment,Total Sub Injured In Incident,Subject Injured In Incident,Subject Injured Prior To Incident,Perceived Condition Of Subject,Subject Actions,Subject Resistance,Subject Medical Treatment,Subject Injury Type,Subject Arrested,Reason Subject Not Arrested,Subject Type,Subject Age,Subject Race/Ethnicity,Subject Gender,Force Type,Incident Year,KEEP/DROP
6768,1089375,Hudson,Hoboken PD,David Chicara,222981.0000,HUDSON-HOBOKEN PD-24-29560,UOF24-4-92,24-29560,4/28/2024,Not Provided,True,"Hoboken, Hudson County",Indoors,Clear,Yes,Body Worn,Artificial,Hospital,Assisting another officer,Dispatched,NaN,28,Not Provided,Not Provided,Other,Not injured,FALSE,NaN,NaN,0.0000,No,Yes,Under influence of alcohol/drugs/both,"Resisted arrest/police officer control, Attemp...",Active Assailant,NaN,NaN,TRUE,NaN,Person,36,Black or African American,Male,Used arms/hands,2024,KEEP
28193,1090120,Hudson,Hoboken PD,Nasir Willey,362175.0000,HUDSON-HOBOKEN PD-24-29560,UOF24-4-101,24-29560,4/28/2024,Not Provided,True,"Hoboken, Hudson County",Indoors,Clear,Yes,Body Worn,Artificial,Hospital,"Assisting another officer, Assault",Dispatched,NaN,21,Not Provided,Not Provided,Other,Complaint of pain,TRUE,Hospital,Treated and Released,0.0000,No,Yes,No unusual condition noted,"Resisted arrest/police officer control, Attemp...",Active Assailant,NaN,NaN,TRUE,NaN,Person,36,Black or African American,Male,Used arms/hands,2024,KEEP
29643,1090660,Hudson,Hoboken PD,Ramon Calderon,18648.0000,HUDSON-HOBOKEN PD-24-29560,UOF24-4-106,24-29560,4/27/2024,Not Provided,True,"Hoboken, Hudson County","Indoors, Outdoors",Clear,Yes,Body Worn,Darkness,"Street, Police Station","Assault, Disturbance (drinking, fighting, diso...",Citizen Initiated,NaN,48,Not Provided,Not Provided,Other,Not injured,FALSE,NaN,NaN,1.0000,Yes,No,"Other unusual condition noted, Under influence...","Attempt to commit crime, Attempt to self-harm,...","Threatening Assailant, Active Assailant, Activ...",Hospital,Complaint of pain,TRUE,NaN,Person,36,Black or African American,Male,Used arms/hands,2024,KEEP
33384,1090125,Hudson,Hoboken PD,Joshua Campoverde,18655.0000,HUDSON-HOBOKEN PD-24-29560,UOF24-4-102,24-29560,4/28/2024,Not Provided,True,"Hoboken, Hudson County",Indoors,Clear,Yes,Body Worn,Artificial,Hospital,"MV/Traffic Stop, Assault, Possession of CDS, T...",Dispatched,NaN,28,Not Provided,Not Provided,Other,Not injured,FALSE,NaN,NaN,0.0000,No,Yes,No unusual condition noted,"Biting, Attempt to self-harm, Prevent harm to ...",Active Assailant,NaN,NaN,TRUE,NaN,Person,36,Black or African American,Male,Used arms/hands,2024,KEEP
43565,1089316,Hudson,Hoboken PD,Alexander Miller,362179.0000,HUDSON-HOBOKEN PD-24-29560,UOF24-4-90,24-29560,4/28/2024,Not Provided,True,"Hoboken, Hudson County",Indoors,NaN,Yes,"Body Worn, Station House",Artificial,Police Station,"Assault, Assisting another officer",Dispatched,NaN,30,White,Officer,Male,Not injured,FALSE,NaN,NaN,1.0000,Yes,Yes,No unusual condition noted,"Attempt to self-harm, Resisted arrest/police o...",Active Assailant,Hospital,"Contusion/bruise, Abrasion/Laceration/Puncture",TRUE,NaN,Person,36,Black or African American,Male,Used take down on,2024,KEEP
47527,1090129,Hudson,Hoboken PD,Tyler Soto,18668.0000,HUDSON-HOBOKEN PD-24-29560,UOF24-4-103,24-29560,4/28/2024,Not Provided,True,"Hoboken, Hudson County",Indoors,Clear,Yes,"Body Worn, Station House",Artificial,Police Station,Assault,Dispatched,NaN,35,Not Provided,Not Provided,Other,Not injured,FALSE,NaN,NaN,1.0000,Yes,Yes,No unusual condition noted,"Resisted arrest/police off

In [50]:
nj_df[nj_df["Subject Gender"].str.contains(",", regex=False)].head().T

,54,179,193,325,396
Form ID,1671580,164927,1220844,1315411,844380
County,Union,Morris,Mercer,Bergen,Bergen
Agency Name,Roselle PD,Hanover Twp PD,Trenton PD,Fort Lee PD,Fair Lawn PD
Officer Name,Bruno Felicia,Robert Carpenter,Matthew Lupinacci,Doglas Pichardo,Matthew Hearon
User ID,224205.0000,26937.0000,190327.0000,474103.0000,128796.0000
Incident ID,UNION-ROSELLE PD-2025-121390,MORRIS-HANOVER TWP PD-21-28256,MERCER-TRENTON PD-24-008761,BERGEN-FORT LEE PD-I-2024-053068,BERGEN-FAIR LAWN PD-23-15552
Report Number,UOF25-7-86,UOF21-11-18,UOF24-8-287,UOF24-10-56,UOF23-10-12
Incident Case Number,2025-121390,21-28256,24-008761,I-2024-053068,23-15552
Incident Date,7/8/2025,10/28/2021,8/5/2024,10/17/2024,10/19/2023
Other Officer Involved,Not Provided,TRUE,Not Provided,Not Provided,Not Provided


In [51]:
for col in ["Subject Gender", "Subject Race/Ethnicity", "Subject Age", "Subject Injured Prior To Incident", 
            "Perceived Condition Of Subject", "Subject Actions", "Subject Resistance", "Subject Arrested", "Force Type"]:
    print(f"{col} : {nj_df[nj_df[col].str.contains(",", regex=False)].shape}")

Subject Gender : (1030, 46)
Subject Race/Ethnicity : (1030, 46)
Subject Age : (1030, 46)
Subject Injured Prior To Incident : (1847, 46)
Perceived Condition Of Subject : (12120, 46)
Subject Actions : (51047, 46)
Subject Resistance : (35252, 46)
Subject Arrested : (1030, 46)
Force Type : (12329, 46)


In [52]:
def expand_nj_subjects(df, delim=','):
    """
    For rows where multiple subjects are encoded with , as a delimiter,
    split into one row per subject.
    
    Subject Race/Ethnicity and Subject Gender are the anchors — the number
    of tokens they produce (split by |) defines how many rows a record expands to.
    
    For Subject Age, Subject Injured Prior To Incident, Perceived Condition Of Subject, Subject Actions, Subject Resistance, Subject Arrested:
      - if the column has the same number of |-delimited tokens as the anchor, split them too
      - otherwise, carry the original value into every expanded row
    """
    anchor_cols = ["Subject Race/Ethnicity", "Subject Gender"]
    conditional_cols = [
        "Subject Age", "Subject Injured In Incident", "Subject Injured Prior To Incident", 
        "Perceived Condition Of Subject", "Subject Actions", 
        "Subject Resistance", "Subject Arrested", 
        "Subject Medical Treatment", "Subject Injury Type"
    ]

    expanded_rows = []

    for idx, row in df.iterrows():
        # Split anchors by deliminiator
        race_val = row[anchor_cols[0]]
        gender_val = row[anchor_cols[1]]

        race_tokens = [t.strip() for t in str(race_val).split(delim)] if pd.notna(race_val) else [race_val]
        gender_tokens = [t.strip() for t in str(gender_val).split(delim)] if pd.notna(gender_val) else [gender_val]

        n = len(gender_tokens)

        # If gender token count mismatches race, broadcast gender to all rows
        if len(race_tokens) != n:
            race_tokens = [race_val] * n

        # Pre-split conditional columns
        cond_splits = {}
        for col in conditional_cols:
            val = row[col]
            if pd.notna(val):
                tokens = [t.strip() for t in str(val).split(delim)]
                cond_splits[col] = tokens if len(tokens) == n else None
            else:
                cond_splits[col] = None

        for i in range(n):
            new_row = row.copy()
            new_row[anchor_cols[0]] = race_tokens[i]
            new_row[anchor_cols[1]] = gender_tokens[i]

            for col in conditional_cols:
                if cond_splits[col] is not None:
                    new_row[col] = cond_splits[col][i]
                # else: leave original broadcast value as-is

            expanded_rows.append(new_row)

    return pd.DataFrame(expanded_rows).reset_index(drop=True)


print("nj_df shape before expansion:", nj_df.shape)
nj_df = expand_nj_subjects(nj_df)
print("nj_df shape after expansion:", nj_df.shape)


nj_df shape before expansion: (96688, 46)
nj_df shape after expansion: (97877, 46)


## Ohio

In [53]:
ohio_path = './data/Ohio/OIBRSUOF.csv'
ohio_df = pd.read_csv(ohio_path, skiprows=1, encoding='latin-1')
ohio_df.drop(columns=["Unnamed: 16"], inplace=True)
ohio_df = reduce_memory_usage(ohio_df, verbose=True)
ohio_df["IncidentNumber"] = ohio_df["IncidentNumber"].str.replace("\t", "", regex=False)
print(ohio_df.shape)
ohio_df.head()


Memory usage before: 2.98 MB
Memory usage after: 2.98 MB
Reduced by 0.0%
(2993, 16)


,AgencyName,IncidentNumber,IncidentDate,InitialContactCircumstances,LocationType,SubjectResistanceTypes,SubjectArmed/BelievedToBeArmed,SubjectRaceEthnicity,SubjectGender,SubjectInjuries,SubjectImpairments,OfficerType,OfficerResponse/ForceTypes,OfficerRaceEthnicity,OfficerGender,OfficerInjuries
0,Akron,202400000028,1/1/2024,Unknown and is Unlikely to Ever be Known,Unknown And Is Unlikely To Ever Be Known,Failing to Comply to Verbal Commands | Pulling...,No,White,Male,NaN,NaN,Law Enforcement,Other Force Type Used | Restraining Hold | Oth...,White | White,Female | Male,Apparent Minor Injury | None
1,Ravenna,2240010006,1/1/2024,Other Circumstances,Government Office,Verbally Threatening Others,No,White,Male,NaN,Alcohol Impairment,Law Enforcement,Restraining Hold,White | White,Male | Male,NaN
2,Holland,24-000001-1,1/1/2024,Responding to Other Unlawful or Suspicious Act...,Single Family Home,Failing to Comply to Verbal Commands | Resisti...,No,Black or African American | Black or African A...,Male | Male,NaN,NaN,Law Enforcement,Balance Displacement | Electronic Control Devi...,White | White,Male | Male,NaN
3,Woodlawn,202400000001,1/1/2024,Traffic Stop,Single Family Home,Firearm Displayed at an Officer or Another,Yes,Black or African American,Female,NaN,NaN,Law Enforcement,Other Force Type Used,Black or African American | White,Male | Male,NaN
4,Cleveland,2024-000866,1/1/2024,Unknown and is Unlikely to Ever be Known,Unknown And Is Unlikely To Ever Be Known,Pulling Away | Punching/Kicking Officer or Ano...,Unknown and is Unlikely to Ever be Known,Unknown/Not Reported | Unknown/Not Reported | ...,Unknown/Not Reported | Female | Unknown/Not Re...,NaN,Alcohol Impairment,Law Enforcement,"Chemical Agent/Spray (Oleoresin Capsicum, Pepp...",Black or African American,Male,NaN


In [54]:
ohio_df[lambda x: x["SubjectResistanceTypes"].str.contains("|", regex=False)].shape

(2107, 16)

In [55]:
ohio_df[lambda x: x["SubjectResistanceTypes"].str.contains("|", regex=False)].head().T

,0,2,4,5,6
AgencyName,Akron,Holland,Cleveland,Greenhills,Ohio State Highway Patrol
IncidentNumber,202400000028,24-000001-1,2024-000866,202400000001,247700010377
IncidentDate,1/1/2024,1/1/2024,1/1/2024,1/1/2024,1/1/2024
InitialContactCircumstances,Unknown and is Unlikely to Ever be Known,Responding to Other Unlawful or Suspicious Act...,Unknown and is Unlikely to Ever be Known,Domestic Disturbance,Traffic Stop
LocationType,Unknown And Is Unlikely To Ever Be Known,Single Family Home,Unknown And Is Unlikely To Ever Be Known,Single Family Home,Street
SubjectResistanceTypes,Failing to Comply to Verbal Commands | Pulling...,Failing to Comply to Verbal Commands | Resisti...,Pulling Away | Punching/Kicking Officer or Ano...,Failing to Comply to Verbal Commands | Nonviol...,Deadweight | Failing to Comply to Verbal Commands
SubjectArmed/BelievedToBeArmed,No,No,Unknown and is Unlikely to Ever be Known,No,Yes
SubjectRaceEthnicity,White,Black or African American | Black or African A...,Unknown/Not Reported | Unknown/Not Reported | ...,White,Black or African American
SubjectGender,Male,Male | Male,Unknown/Not Reported | Female | Unknown/Not Re...,Male,Male
SubjectInjuries,NaN,NaN,NaN,NaN,Apparent Minor Injury


In [56]:
ohio_df[lambda x: x["SubjectArmed/BelievedToBeArmed"].str.contains("|", regex=False)].shape

(14, 16)

In [57]:
ohio_df[lambda x: x["SubjectRaceEthnicity"].str.contains("|", regex=False)].shape

(217, 16)

In [58]:
ohio_df[lambda x: x["SubjectRaceEthnicity"].str.contains("|", regex=False)].head(2).T

,2,4
AgencyName,Holland,Cleveland
IncidentNumber,24-000001-1,2024-000866
IncidentDate,1/1/2024,1/1/2024
InitialContactCircumstances,Responding to Other Unlawful or Suspicious Act...,Unknown and is Unlikely to Ever be Known
LocationType,Single Family Home,Unknown And Is Unlikely To Ever Be Known
SubjectResistanceTypes,Failing to Comply to Verbal Commands | Resisti...,Pulling Away | Punching/Kicking Officer or Ano...
SubjectArmed/BelievedToBeArmed,No,Unknown and is Unlikely to Ever be Known
SubjectRaceEthnicity,Black or African American | Black or African A...,Unknown/Not Reported | Unknown/Not Reported | ...
SubjectGender,Male | Male,Unknown/Not Reported | Female | Unknown/Not Re...
SubjectInjuries,NaN,NaN


In [59]:
ohio_df[lambda x: x["SubjectRaceEthnicity"].str.contains("|", regex=False)].loc[2]["SubjectResistanceTypes"]

'Failing to Comply to Verbal Commands | Resisting Being Handcuffed or Arrest'

In [60]:
ohio_df[lambda x: x["SubjectRaceEthnicity"].str.contains("|", regex=False)].loc[2]["OfficerResponse/ForceTypes"]

'Balance Displacement | Electronic Control Device (ECD) Discharged'

In [61]:
def expand_ohio_subjects(df):
    """
    For rows where multiple subjects are encoded with | as a delimiter,
    split into one row per subject.
    
    SubjectRaceEthnicity and SubjectGender are the anchors — the number
    of tokens they produce (split by |) defines how many rows a record expands to.
    
    For SubjectArmed/BelievedToBeArmed, SubjectInjuries, SubjectImpairments:
      - if the column has the same number of |-delimited tokens as the anchor, split them too
      - otherwise, carry the original value into every expanded row
    """
    anchor_cols = ["SubjectRaceEthnicity", "SubjectGender"]
    conditional_cols = [
        "SubjectArmed/BelievedToBeArmed",
        "SubjectInjuries",
        "SubjectImpairments",
    ]

    expanded_rows = []

    for idx, row in df.iterrows():
        # Split anchors by |
        race_val = row["SubjectRaceEthnicity"]
        gender_val = row["SubjectGender"]

        race_tokens = [t.strip() for t in str(race_val).split("|")] if pd.notna(race_val) else [race_val]
        gender_tokens = [t.strip() for t in str(gender_val).split("|")] if pd.notna(gender_val) else [gender_val]

        n = len(gender_tokens)

        # If gender token count mismatches race, broadcast gender to all rows
        if len(race_tokens) != n:
            race_tokens = [race_val] * n

        # Pre-split conditional columns
        cond_splits = {}
        for col in conditional_cols:
            val = row[col]
            if pd.notna(val):
                tokens = [t.strip() for t in str(val).split("|")]
                cond_splits[col] = tokens if len(tokens) == n else None
            else:
                cond_splits[col] = None

        for i in range(n):
            new_row = row.copy()
            new_row["SubjectRaceEthnicity"] = race_tokens[i]
            new_row["SubjectGender"] = gender_tokens[i]

            for col in conditional_cols:
                if cond_splits[col] is not None:
                    new_row[col] = cond_splits[col][i]
                # else: leave original broadcast value as-is

            expanded_rows.append(new_row)

    return pd.DataFrame(expanded_rows).reset_index(drop=True)


print("ohio_df shape before expansion:", ohio_df.shape)
ohio_df = expand_ohio_subjects(ohio_df)
print("ohio_df shape after expansion:", ohio_df.shape)


ohio_df shape before expansion: (2993, 16)
ohio_df shape after expansion: (3258, 16)


In [62]:
ohio_df[lambda x: x.IncidentNumber == "202400001349"]


,AgencyName,IncidentNumber,IncidentDate,InitialContactCircumstances,LocationType,SubjectResistanceTypes,SubjectArmed/BelievedToBeArmed,SubjectRaceEthnicity,SubjectGender,SubjectInjuries,SubjectImpairments,OfficerType,OfficerResponse/ForceTypes,OfficerRaceEthnicity,OfficerGender,OfficerInjuries
969,Blue Ash,202400001349,12/28/2024,Traffic Stop,Street,Deadweight | Failing to Comply to Verbal Commands,No,White | Hispanic or Latino | White | Hispanic ...,Male,None,NaN,Law Enforcement,Other Empty Hand Technique Used,Black or African American | White,Male | Male,NaN
970,Blue Ash,202400001349,12/28/2024,Traffic Stop,Street,Deadweight | Failing to Comply to Verbal Commands,No,White | Hispanic or Latino | White | Hispanic ...,Male,Unknown and is unlikely to ever be known,NaN,Law Enforcement,Other Empty Hand Technique Used,Black or African American | White,Male | Male,NaN


In [63]:
ohio_df.head()

,AgencyName,IncidentNumber,IncidentDate,InitialContactCircumstances,LocationType,SubjectResistanceTypes,SubjectArmed/BelievedToBeArmed,SubjectRaceEthnicity,SubjectGender,SubjectInjuries,SubjectImpairments,OfficerType,OfficerResponse/ForceTypes,OfficerRaceEthnicity,OfficerGender,OfficerInjuries
0,Akron,202400000028,1/1/2024,Unknown and is Unlikely to Ever be Known,Unknown And Is Unlikely To Ever Be Known,Failing to Comply to Verbal Commands | Pulling...,No,White,Male,NaN,NaN,Law Enforcement,Other Force Type Used | Restraining Hold | Oth...,White | White,Female | Male,Apparent Minor Injury | None
1,Ravenna,2240010006,1/1/2024,Other Circumstances,Government Office,Verbally Threatening Others,No,White,Male,NaN,Alcohol Impairment,Law Enforcement,Restraining Hold,White | White,Male | Male,NaN
2,Holland,24-000001-1,1/1/2024,Responding to Other Unlawful or Suspicious Act...,Single Family Home,Failing to Comply to Verbal Commands | Resisti...,No,Black or African American,Male,NaN,NaN,Law Enforcement,Balance Displacement | Electronic Control Devi...,White | White,Male | Male,NaN
3,Holland,24-000001-1,1/1/2024,Responding to Other Unlawful or Suspicious Act...,Single Family Home,Failing to Comply to Verbal Commands | Resisti...,No,Black or African American,Male,NaN,NaN,Law Enforcement,Balance Displacement | Electronic Control Devi...,White | White,Male | Male,NaN
4,Woodlawn,202400000001,1/1/2024,Traffic Stop,Single Family Home,Firearm Displayed at an Officer or Another,Yes,Black or African American,Female,NaN,NaN,Law Enforcement,Other Force Type Used,Black or African American | White,Male | Male,NaN


In [64]:
ohio_df.loc[1050]

AgencyName                                                                   Medina
IncidentNumber                                                          20-24004717
IncidentDate                                                              12/9/2024
InitialContactCircumstances                                    Domestic Disturbance
LocationType                                                      Multiple Dwelling
SubjectResistanceTypes            Attempt to Escape/Flee From Custody | Deadweig...
SubjectArmed/BelievedToBeArmed                                                   No
SubjectRaceEthnicity                                      Black or African American
SubjectGender                                                                  Male
SubjectInjuries                                                                 NaN
SubjectImpairments                                               Alcohol Impairment
OfficerType                                                         Law Enfo

In [65]:
ohio_df["OfficerResponse/ForceTypes"].str.split("|").explode().str.strip().value_counts()

OfficerResponse/ForceTypes
Restraining Hold                                                1421
Take Down                                                       1088
Other Force Type Used                                            919
Balance Displacement                                             628
Other Empty Hand Technique Used                                  625
Electronic Control Device (ECD) Discharged                       530
Pressure Point                                                   206
Chemical Agent/Spray (Oleoresin Capsicum, Pepper, etc.) Used     122
Canine                                                            39
Other Weapon (non-firearm) Used                                   34
Unknown and is Unlikely to Ever be Known                          23
Baton Used                                                        22
Pending Further Investigation                                     19
Handgun Fired                                                      9
Vehicle

## New York

In [66]:
ny_uof_path = './data/NewYork/UOF Case Level File_Nov2020-Dec2024.xlsx'
ny_uof_df = pd.read_excel(ny_uof_path)
ny_uof_df.head()


,Response ID\n(5/16/25 File),Agency Name,Incident Date,City/town/village (location Where Incident Occurred),County (location where incident occurred),Circumstance,Officer 1 - Use of Force Type,Officer 1 Age,Officer 1 Sex,Officer 1 Race,Officer 1 Ethnicity,Officer 2 - Use of Force Type,Officer 2 Age,Officer 2 Sex,Officer 2 Race,Officer 2 Ethnicity,Officer 3 - Use of Force Type,Officer 3 Age,Officer 3 Sex,Officer 3 Race,Officer 3 Ethnicity,Officer 4 - Use of Force Type,Officer 4 Age,Officer 4 Sex,Officer 4 Race,Officer 4 Ethnicity,Officer 5 - Use of Force Type,Officer 5 Age,Officer 5 Sex,Officer 5 Race,Officer 5 Ethnicity,Officer 6 - Use of Force Type,Officer 6 Age,Officer 6 Sex,Officer 6 Race,Officer 6 Ethnicity,Officer 7 - Use of Force Type,Officer 7 Age,Officer 7 Sex,Officer 7 Race,Officer 7 Ethnicity,Officer 8 - Use of Force Type,Officer 8 Age,Officer 8 Sex,Officer 8 Race,Officer 8 Ethnicity,Officer 9 - Use of Force Type,Officer 9 Age,Officer 9 Sex,Officer 9 Race,Officer 9 Ethnicity,Officer 10 - Use of Force Type,Officer 10 Age,Officer 10 Sex,Officer 10 Race,Officer 10 Ethnicity,Officer 11 - Use of Force Type,Officer 11 Age,Officer 11 Sex,Officer 11 Race,Officer 11 Ethnicity,Officer 12 - Use of Force Type,Officer 12 Age,Officer 12 Sex,Officer 12 Race,Officer 12 Ethnicity,Officer 13 - Use of Force Type,Officer 13 Age,Officer 13 Sex,Officer 13 Race,Officer 13 Ethnicity,Officer 14 - Use of Force Type,Officer 14 Age,Officer 14 Sex,Officer 14 Race,Officer 14 Ethnicity,Officer 15 - Use of Force Type,Officer 15 Age,Officer 15 Sex,Officer 15 Race,Officer 15 Ethnicity,Officer 16 - Use of Force Type,Officer 16 Age,Officer 16 Sex,Officer 16 Race,Officer 16 Ethnicity,Subject 1 Age,Subject 1 Sex,Subject 1 Race,Subject 1 Ethnicity,Subject 2 Age,Subject 2 Sex,Subject 2 Race,Subject 2 Ethnicity,Subject 3 Age,Subject 3 Sex,Subject 3 Race,Subject 3 Ethnicity,Subject 4 Age,Subject 4 Sex,Subject 4 Race,Subject 4 Ethnicity,Subject 5 Age,Subject 5 Sex,Subject 5 Race,Subject 5 Ethnicity,Subject 6 Age,Subject 6 Sex,Subject 6 Race,Subject 6 Ethnicity,Subject 7 Age,Subject 7 Sex,Subject 7 Race,Subject 7 Ethnicity,Subject 8 Age,Subject 8 Sex,Subject 8 Race,Subject 8 Ethnicity,Subject 9 Age,Subject 9 Sex,Subject 9 Race,Subject 9 Ethnicity,Subject 10 Age,Subject 10 Sex,Subject 10 Race,Subject 10 Ethnicity,Subject 11 Age,Subject 11 Sex,Subject 11 Race,Subject 11 Ethnicity,Subject 12 Age,Subject 12 Sex,Subject 12 Race,Subject 12 Ethnicity,Subject 13 Age,Subject 13 Sex,Subject 13 Race,Subject 13 Ethnicity,Subject 14 Age,Subject 14 Sex,Subject 14 Race,Subject 14 Ethnicity,Subject 15 Age,Subject 15 Sex,Subject 15 Race,Subject 15 Ethnicity,Subject 16 Age,Subject 16 Sex,Subject 16 Race,Subject 16 Ethnicity,Subject 17 Age,Subject 17 Sex,Subject 17 Race,Subject 17 Ethnicity,Subject 18 Age,Subject 18 Sex,Subject 18 Race,Subject 18 Ethnicity,Subject 19 Age,Subject 19 Sex,Subject 19 Race,Subject 19 Ethnicity,Subject 20 Age,Subject 20 Sex,Subject 20 Race,Subject 20 Ethnicity,Subject 21 Age,Subject 21 Sex,Subject 21 Race,Subject 21 Ethnicity,Subject 22 Age,Subject 22 Sex,Subject 22 Race,Subject 22 Ethnicity,Subject 23 Age,Subject 23 Sex,Subject 23 Race,Subject 23 Ethnicity,Subject 24 Age,Subject 24 Sex,Subject 24 Race,Subject 24 Ethnicity,Subject 25 Age,Subject 25 Sex,Subject 25 Race,Subject 25 Ethnicity
0,1,Akron Vg PD,2020-12-22,Akron,Erie,Response to unlawful or suspicious activity,Pointed Firearm,27.0000,Male,White,Non-Hispanic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49.0000,Male,White,Non-Hispanic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

In [67]:
ny_uof_df.shape

(34837, 186)

In [68]:
(
    ny_uof_df
        [lambda x: x["Incident Date"].between(datetime(2024,1,1), datetime(2024,12,31), inclusive="both")]
        .sort_values(by="Incident Date")
)

,Response ID\n(5/16/25 File),Agency Name,Incident Date,City/town/village (location Where Incident Occurred),County (location where incident occurred),Circumstance,Officer 1 - Use of Force Type,Officer 1 Age,Officer 1 Sex,Officer 1 Race,Officer 1 Ethnicity,Officer 2 - Use of Force Type,Officer 2 Age,Officer 2 Sex,Officer 2 Race,Officer 2 Ethnicity,Officer 3 - Use of Force Type,Officer 3 Age,Officer 3 Sex,Officer 3 Race,Officer 3 Ethnicity,Officer 4 - Use of Force Type,Officer 4 Age,Officer 4 Sex,Officer 4 Race,Officer 4 Ethnicity,Officer 5 - Use of Force Type,Officer 5 Age,Officer 5 Sex,Officer 5 Race,Officer 5 Ethnicity,Officer 6 - Use of Force Type,Officer 6 Age,Officer 6 Sex,Officer 6 Race,Officer 6 Ethnicity,Officer 7 - Use of Force Type,Officer 7 Age,Officer 7 Sex,Officer 7 Race,Officer 7 Ethnicity,Officer 8 - Use of Force Type,Officer 8 Age,Officer 8 Sex,Officer 8 Race,Officer 8 Ethnicity,Officer 9 - Use of Force Type,Officer 9 Age,Officer 9 Sex,Officer 9 Race,Officer 9 Ethnicity,Officer 10 - Use of Force Type,Officer 10 Age,Officer 10 Sex,Officer 10 Race,Officer 10 Ethnicity,Officer 11 - Use of Force Type,Officer 11 Age,Officer 11 Sex,Officer 11 Race,Officer 11 Ethnicity,Officer 12 - Use of Force Type,Officer 12 Age,Officer 12 Sex,Officer 12 Race,Officer 12 Ethnicity,Officer 13 - Use of Force Type,Officer 13 Age,Officer 13 Sex,Officer 13 Race,Officer 13 Ethnicity,Officer 14 - Use of Force Type,Officer 14 Age,Officer 14 Sex,Officer 14 Race,Officer 14 Ethnicity,Officer 15 - Use of Force Type,Officer 15 Age,Officer 15 Sex,Officer 15 Race,Officer 15 Ethnicity,Officer 16 - Use of Force Type,Officer 16 Age,Officer 16 Sex,Officer 16 Race,Officer 16 Ethnicity,Subject 1 Age,Subject 1 Sex,Subject 1 Race,Subject 1 Ethnicity,Subject 2 Age,Subject 2 Sex,Subject 2 Race,Subject 2 Ethnicity,Subject 3 Age,Subject 3 Sex,Subject 3 Race,Subject 3 Ethnicity,Subject 4 Age,Subject 4 Sex,Subject 4 Race,Subject 4 Ethnicity,Subject 5 Age,Subject 5 Sex,Subject 5 Race,Subject 5 Ethnicity,Subject 6 Age,Subject 6 Sex,Subject 6 Race,Subject 6 Ethnicity,Subject 7 Age,Subject 7 Sex,Subject 7 Race,Subject 7 Ethnicity,Subject 8 Age,Subject 8 Sex,Subject 8 Race,Subject 8 Ethnicity,Subject 9 Age,Subject 9 Sex,Subject 9 Race,Subject 9 Ethnicity,Subject 10 Age,Subject 10 Sex,Subject 10 Race,Subject 10 Ethnicity,Subject 11 Age,Subject 11 Sex,Subject 11 Race,Subject 11 Ethnicity,Subject 12 Age,Subject 12 Sex,Subject 12 Race,Subject 12 Ethnicity,Subject 13 Age,Subject 13 Sex,Subject 13 Race,Subject 13 Ethnicity,Subject 14 Age,Subject 14 Sex,Subject 14 Race,Subject 14 Ethnicity,Subject 15 Age,Subject 15 Sex,Subject 15 Race,Subject 15 Ethnicity,Subject 16 Age,Subject 16 Sex,Subject 16 Race,Subject 16 Ethnicity,Subject 17 Age,Subject 17 Sex,Subject 17 Race,Subject 17 Ethnicity,Subject 18 Age,Subject 18 Sex,Subject 18 Race,Subject 18 Ethnicity,Subject 19 Age,Subject 19 Sex,Subject 19 Race,Subject 19 Ethnicity,Subject 20 Age,Subject 20 Sex,Subject 20 Race,Subject 20 Ethnicity,Subject 21 Age,Subject 21 Sex,Subject 21 Race,Subject 21 Ethnicity,Subject 22 Age,Subject 22 Sex,Subject 22 Race,Subject 22 Ethnicity,Subject 23 Age,Subject 23 Sex,Subject 23 Race,Subject 23 Ethnicity,Subject 24 Age,Subject 24 Sex,Subject 24 Race,Subject 24 Ethnicity,Subject 25 Age,Subject 25 Sex,Subject 25 Race,Subject 25 Ethnicity
26977,26978,Syracuse City PD,2024-01-01,Syracuse,Onondaga,Response to unlawful or suspicious activity,Pointed Electronic Control Weapon,26.0000,Male,White,Non-Hispanic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.0000,Male,Black,Unknown,NaN,Male,Black,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

In [69]:
ny_uof_df[~ny_uof_df["Subject 25 Race"].isna()]

,Response ID\n(5/16/25 File),Agency Name,Incident Date,City/town/village (location Where Incident Occurred),County (location where incident occurred),Circumstance,Officer 1 - Use of Force Type,Officer 1 Age,Officer 1 Sex,Officer 1 Race,Officer 1 Ethnicity,Officer 2 - Use of Force Type,Officer 2 Age,Officer 2 Sex,Officer 2 Race,Officer 2 Ethnicity,Officer 3 - Use of Force Type,Officer 3 Age,Officer 3 Sex,Officer 3 Race,Officer 3 Ethnicity,Officer 4 - Use of Force Type,Officer 4 Age,Officer 4 Sex,Officer 4 Race,Officer 4 Ethnicity,Officer 5 - Use of Force Type,Officer 5 Age,Officer 5 Sex,Officer 5 Race,Officer 5 Ethnicity,Officer 6 - Use of Force Type,Officer 6 Age,Officer 6 Sex,Officer 6 Race,Officer 6 Ethnicity,Officer 7 - Use of Force Type,Officer 7 Age,Officer 7 Sex,Officer 7 Race,Officer 7 Ethnicity,Officer 8 - Use of Force Type,Officer 8 Age,Officer 8 Sex,Officer 8 Race,Officer 8 Ethnicity,Officer 9 - Use of Force Type,Officer 9 Age,Officer 9 Sex,Officer 9 Race,Officer 9 Ethnicity,Officer 10 - Use of Force Type,Officer 10 Age,Officer 10 Sex,Officer 10 Race,Officer 10 Ethnicity,Officer 11 - Use of Force Type,Officer 11 Age,Officer 11 Sex,Officer 11 Race,Officer 11 Ethnicity,Officer 12 - Use of Force Type,Officer 12 Age,Officer 12 Sex,Officer 12 Race,Officer 12 Ethnicity,Officer 13 - Use of Force Type,Officer 13 Age,Officer 13 Sex,Officer 13 Race,Officer 13 Ethnicity,Officer 14 - Use of Force Type,Officer 14 Age,Officer 14 Sex,Officer 14 Race,Officer 14 Ethnicity,Officer 15 - Use of Force Type,Officer 15 Age,Officer 15 Sex,Officer 15 Race,Officer 15 Ethnicity,Officer 16 - Use of Force Type,Officer 16 Age,Officer 16 Sex,Officer 16 Race,Officer 16 Ethnicity,Subject 1 Age,Subject 1 Sex,Subject 1 Race,Subject 1 Ethnicity,Subject 2 Age,Subject 2 Sex,Subject 2 Race,Subject 2 Ethnicity,Subject 3 Age,Subject 3 Sex,Subject 3 Race,Subject 3 Ethnicity,Subject 4 Age,Subject 4 Sex,Subject 4 Race,Subject 4 Ethnicity,Subject 5 Age,Subject 5 Sex,Subject 5 Race,Subject 5 Ethnicity,Subject 6 Age,Subject 6 Sex,Subject 6 Race,Subject 6 Ethnicity,Subject 7 Age,Subject 7 Sex,Subject 7 Race,Subject 7 Ethnicity,Subject 8 Age,Subject 8 Sex,Subject 8 Race,Subject 8 Ethnicity,Subject 9 Age,Subject 9 Sex,Subject 9 Race,Subject 9 Ethnicity,Subject 10 Age,Subject 10 Sex,Subject 10 Race,Subject 10 Ethnicity,Subject 11 Age,Subject 11 Sex,Subject 11 Race,Subject 11 Ethnicity,Subject 12 Age,Subject 12 Sex,Subject 12 Race,Subject 12 Ethnicity,Subject 13 Age,Subject 13 Sex,Subject 13 Race,Subject 13 Ethnicity,Subject 14 Age,Subject 14 Sex,Subject 14 Race,Subject 14 Ethnicity,Subject 15 Age,Subject 15 Sex,Subject 15 Race,Subject 15 Ethnicity,Subject 16 Age,Subject 16 Sex,Subject 16 Race,Subject 16 Ethnicity,Subject 17 Age,Subject 17 Sex,Subject 17 Race,Subject 17 Ethnicity,Subject 18 Age,Subject 18 Sex,Subject 18 Race,Subject 18 Ethnicity,Subject 19 Age,Subject 19 Sex,Subject 19 Race,Subject 19 Ethnicity,Subject 20 Age,Subject 20 Sex,Subject 20 Race,Subject 20 Ethnicity,Subject 21 Age,Subject 21 Sex,Subject 21 Race,Subject 21 Ethnicity,Subject 22 Age,Subject 22 Sex,Subject 22 Race,Subject 22 Ethnicity,Subject 23 Age,Subject 23 Sex,Subject 23 Race,Subject 23 Ethnicity,Subject 24 Age,Subject 24 Sex,Subject 24 Race,Subject 24 Ethnicity,Subject 25 Age,Subject 25 Sex,Subject 25 Race,Subject 25 Ethnicity
3025,3026,Buffalo City PD,2024-10-14,Buffalo,Erie,Service of a warrant,Pointed Firearm,55.0000,Male,White,Non-Hispanic,Pointed Firearm,40.0000,Male,White,Hispanic,Pointed Firearm,45.0000,Male,White,Non-Hispanic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.0000,Female,Black,Non-Hispanic,56.0000,Female,Black,Non-Hispanic,32.0000,Female,Black,Non-Hispanic,35.0000,Female,Black,Unknown,26.0000,Male,Black,Non-Hispanic,35.0000,Female,Black,Unknown,36.0000,Fema

In [70]:
ny_uof_df = ny_uof_df.reset_index(drop=True).rename_axis("incident_id").reset_index()
# Identify non-subject columns
id_cols = [
    col for col in ny_uof_df.columns
    if not re.match(r"Subject \d+ (Race|Ethnicity|Age|Sex)$", col)
]

# Rename subject columns to the format wide_to_long expects: "race1", "race2", etc.
rename_map = {}
for col in ny_uof_df.columns:
    m = re.match(r"Subject (\d+) (Race|Ethnicity|Age|Sex)$", col)
    if m:
        num, attr = m.groups()
        rename_map[col] = f"{attr.lower()}{num}"

ny_uof_long = (
    ny_uof_df
    .rename(columns=rename_map)
    .pipe(pd.wide_to_long,
          stubnames=["race", "ethnicity", "age", "sex"],
          i=id_cols,
          j="subject_number")
    .reset_index()
    .rename(columns={"race": "subject_race", "ethnicity": "subject_ethnicity",
                     "age": "subject_age", "sex": "subject_sex"})
    .loc[lambda df: ~(
        df[["subject_race", "subject_ethnicity", "subject_age", "subject_sex"]]
        .isin([""])
        .all(axis=1)
        & df[["subject_race", "subject_ethnicity", "subject_age", "subject_sex"]]
        .isna()
        .all(axis=1)
    )]
    .reset_index(drop=True)
)
subject_cols = ["subject_race", "subject_ethnicity", "subject_age", "subject_sex"]
ny_uof_long = ny_uof_long.dropna(subset=subject_cols, how="all")

In [71]:
ny_uof_long

,incident_id,Response ID\n(5/16/25 File),Agency Name,Incident Date,City/town/village (location Where Incident Occurred),County (location where incident occurred),Circumstance,Officer 1 - Use of Force Type,Officer 1 Age,Officer 1 Sex,Officer 1 Race,Officer 1 Ethnicity,Officer 2 - Use of Force Type,Officer 2 Age,Officer 2 Sex,Officer 2 Race,Officer 2 Ethnicity,Officer 3 - Use of Force Type,Officer 3 Age,Officer 3 Sex,Officer 3 Race,Officer 3 Ethnicity,Officer 4 - Use of Force Type,Officer 4 Age,Officer 4 Sex,Officer 4 Race,Officer 4 Ethnicity,Officer 5 - Use of Force Type,Officer 5 Age,Officer 5 Sex,Officer 5 Race,Officer 5 Ethnicity,Officer 6 - Use of Force Type,Officer 6 Age,Officer 6 Sex,Officer 6 Race,Officer 6 Ethnicity,Officer 7 - Use of Force Type,Officer 7 Age,Officer 7 Sex,Officer 7 Race,Officer 7 Ethnicity,Officer 8 - Use of Force Type,Officer 8 Age,Officer 8 Sex,Officer 8 Race,Officer 8 Ethnicity,Officer 9 - Use of Force Type,Officer 9 Age,Officer 9 Sex,Officer 9 Race,Officer 9 Ethnicity,Officer 10 - Use of Force Type,Officer 10 Age,Officer 10 Sex,Officer 10 Race,Officer 10 Ethnicity,Officer 11 - Use of Force Type,Officer 11 Age,Officer 11 Sex,Officer 11 Race,Officer 11 Ethnicity,Officer 12 - Use of Force Type,Officer 12 Age,Officer 12 Sex,Officer 12 Race,Officer 12 Ethnicity,Officer 13 - Use of Force Type,Officer 13 Age,Officer 13 Sex,Officer 13 Race,Officer 13 Ethnicity,Officer 14 - Use of Force Type,Officer 14 Age,Officer 14 Sex,Officer 14 Race,Officer 14 Ethnicity,Officer 15 - Use of Force Type,Officer 15 Age,Officer 15 Sex,Officer 15 Race,Officer 15 Ethnicity,Officer 16 - Use of Force Type,Officer 16 Age,Officer 16 Sex,Officer 16 Race,Officer 16 Ethnicity,subject_number,subject_race,subject_ethnicity,subject_age,subject_sex
0,0,1,Akron Vg PD,2020-12-22,Akron,Erie,Response to unlawful or suspicious activity,Pointed Firearm,27.0000,Male,White,Non-Hispanic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,White,Non-Hispanic,49.0000,Male
25,1,2,Albany City PD,2020-11-10,Albany,Albany,Other,Used/Deployed Chemical Agent,37.0000,Male,White,Non-Hispanic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Black,Unknown,NaN,Male
50,2,3,Albany City PD,2020-11-20,Albany,Albany,Other,Pointed Firearm,45.0000,Male,White,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Black,Unknown,49.0000,Male
75,3,4,Albany City PD,2020-12-08,Albany,Albany,Executing Arrest,Pointed Electronic Control Weapon,50.0000,Male,White,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Black,Unknown,25.0000,Male
100,4,5,Albany City PD,2021-01-11,Albany,Albany,Executing Arrest,Pointed Firearm,32.0000,Male,White,Non-Hispanic,Pointed Firearm,25.0000,Male,White,Unknown,Pointed Electronic Control Weapon,34.0000,Male,Asian,Unknown,Pointed Electronic Control Weapon,36.0000,Male,White,Non-Hispanic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

In [72]:
ny_uof_long = reduce_memory_usage(ny_uof_long, verbose=True)


Memory usage before: 116.60 MB
Memory usage after: 113.21 MB
Reduced by 2.9%


In [73]:
ny_incidents_path = './data/NewYork/NYPD_Use_of_Force_Incidents_20260211.csv'
ny_incidents_df = pd.read_csv(ny_incidents_path)

ny_subjects_path = './data/NewYork/NYPD_Use_of_Force__Subjects_20260211.csv'
ny_subjects_df = pd.read_csv(ny_subjects_path)

print(ny_incidents_df.shape)
display(ny_incidents_df.head())
print(ny_subjects_df.shape)
ny_subjects_df.head()

(56994, 7)


,TRI Incident Number,ForceType,Occurrence Date,Incident Pct,Patrol Borough,YearMonthShort,BasisForEncounter
0,2024032962162,Physical Force,08/31/2024,13,PBMS,2024 Aug,CRIME/VIOLATION IN PROGRESS
1,2024033962356,Physical Force,08/31/2024,14,PBMS,2024 Aug,CRIME/VIOLATION IN PROGRESS
2,2024090962120,Physical Force,08/31/2024,71,PBBS,2024 Aug,CRIME/VIOLATION IN PROGRESS
3,2024096962122,Physical Force,08/31/2024,77,PBBN,2024 Aug,CRIME/VIOLATION IN PROGRESS
4,2024081962083,Physical Force,08/31/2024,62,PBBS,2024 Aug,CRIME/VIOLATION IN PROGRESS


(58585, 8)


,TRI Incident Number,Subject Injury Level,Subject Injured,Age,Subject Race,Subject Gender,Force Against MOS,Subject Used Force
0,2020020962067,No Injury,N,NaN,BLACK,MALE,Physical Force,Y
1,2020020962143,No Injury,N,NaN,BLACK,MALE,Physical Force,Y
2,2020024962080,No Injury,N,NaN,BLACK,MALE,Physical Force,Y
3,2020025962038,No Injury,N,NaN,BLACK,MALE,Physical Force,Y
4,2020025962049,No Injury,N,NaN,BLACK,MALE,Physical Force,Y


In [74]:
ny_df = pd.merge(left=ny_incidents_df, right=ny_subjects_df, on="TRI Incident Number")
ny_df = reduce_memory_usage(ny_df, verbose=True)

print(ny_df.shape)
ny_df.head()

Memory usage before: 36.71 MB
Memory usage after: 36.15 MB
Reduced by 1.5%
(58585, 14)


,TRI Incident Number,ForceType,Occurrence Date,Incident Pct,Patrol Borough,YearMonthShort,BasisForEncounter,Subject Injury Level,Subject Injured,Age,Subject Race,Subject Gender,Force Against MOS,Subject Used Force
0,2024032962162,Physical Force,08/31/2024,13,PBMS,2024 Aug,CRIME/VIOLATION IN PROGRESS,No Injury,N,17.0000,BLACK,MALE,Physical Force,Y
1,2024033962356,Physical Force,08/31/2024,14,PBMS,2024 Aug,CRIME/VIOLATION IN PROGRESS,No Injury,N,30.0000,BLACK,MALE,Physical Force,Y
2,2024090962120,Physical Force,08/31/2024,71,PBBS,2024 Aug,CRIME/VIOLATION IN PROGRESS,No Injury,N,19.0000,OTHER,MALE,Physical Force,Y
3,2024096962122,Physical Force,08/31/2024,77,PBBN,2024 Aug,CRIME/VIOLATION IN PROGRESS,No Injury,N,38.0000,BLACK,MALE,Physical Force,Y
4,2024081962083,Physical Force,08/31/2024,62,PBBS,2024 Aug,CRIME/VIOLATION IN PROGRESS,Physical Injury,Y,17.0000,HISPANIC,MALE,Physical Force,Y


In [75]:
ny_df["ForceType"].value_counts()

ForceType
Physical Force              48068
Electrical Weapon            8898
OC Spray                      950
Impact Weapon                 438
Firearm                       213
Restraining Mesh Blanket       13
Police Canine                   5
Name: count, dtype: int64

In [76]:
ny_df["Occurrence Date"].dtype

<StringDtype(storage='python', na_value=nan)>

In [77]:
ny_df[lambda x: (x.ForceType == "Firearm") & (x["Occurrence Date"].str.contains("2024"))].shape

(34, 14)

In [78]:
ny_df[lambda x: (x.ForceType == "Restraining Mesh Blanket") & (x["Occurrence Date"].str.contains("2024"))].shape

(0, 14)

## California

In [79]:
cali_dir = './data/California/Final files'
cali_file_names = [item.name for item in Path(cali_dir).iterdir() if item.is_file()]
cali_file_paths = [cali_dir + "/" + name for name in cali_file_names if '.xlsx' in name]
print(len(cali_file_paths))
print(cali_file_paths[0])

65
./data/California/Final files/RIPA Stop Data_San Francisco 2024_final.xlsx


In [80]:
cali_path = cali_dir + "/cali_data.csv"
if not os.path.exists(cali_path):
    dfs = []

    for path, name in tqdm(zip(cali_file_paths, cali_file_names)):
        match = re.search(r'_(.*?)2024', name)
        county = match.group(1) if match else ""
        
        # 1. Read the file
        # 2. .copy() forces immediate de-fragmentation
        # 3. .assign() adds the column to the clean copy
        temp_df = pd.read_excel(path).copy()
        temp_df = temp_df.assign(county_name=county)
        
        dfs.append(temp_df)

    # Final concatenation
    cali_df = pd.concat(dfs, ignore_index=True)
    race_cols_to_drop = [c for c in cali_df.columns if c.startswith('RAE_') and c != 'RAE_FULL']
    gender_cols_to_drop = [c for c in cali_df.columns if c.startswith('G_') and c != 'G_FULL']
    sex_cols_to_drop = ['SOR_LGB']
    pd_cols_to_drop = [c for c in cali_df.columns if c.startswith('PD_') and c != 'PD_FULL']

    cali_df = cali_df.drop(columns=race_cols_to_drop + gender_cols_to_drop + sex_cols_to_drop)

    cali_df = reduce_memory_usage(cali_df, verbose=True)
    cali_df.to_csv(cali_path, index=False)
    del dfs
    gc.collect()

In [81]:
cali_df = pd.read_csv(cali_path)
print(cali_df.shape)
cali_df.head()

/var/folders/8r/qgk4_l1j20gfq28wvs_pxh2r0000gn/T/ipykernel_4510/3596666226.py:1: DtypeWarning: Columns (0: SCHOOL_CODE, 1: SCHOOL_NAME, 2: RFS_PC_CODE, 3: ROS_CITATION_CDS, 4: ROS_IN_FIELD_CITE_RELEASE_CDS, 5: ROS_CUSTODIAL_WOUT_WARRANT_CDS, 6: ROS_VERBAL_WARNING_CDS, 7: ROS_WRITTEN_WARNING_CDS) have mixed types. Specify dtype option on import or set low_memory=False.
  cali_df = pd.read_csv(cali_path)


(5065428, 179)


,DOJ_RECORD_ID,PERSON_NUMBER,AGENCY_ORI,AGENCY_NAME,NON_REPORTING_AGENCY,TIME_OF_STOP,DATE_OF_STOP,STOP_DURATION,LOC_CLOSEST_CITY,SCHOOL_CODE,SCHOOL_NAME,STOP_STUDENT,K12_SCHOOL_GROUNDS,RAE_FULL,G_FULL,SOR_STRAIGHT,AGE,AGE_GROUP,LIMITED_ENGLISH_FLUENCY,PD_FULL,PERSON_UNHOUSED,PASSENGER_IN_VEHICLE,INSIDE_RESIDENCE,WELFARE_WELLNESS_CHECK,TOS_VEHICULAR,TOS_BICYCLE,TOS_PEDESTRIAN,CALL_FOR_SERVICE,REASON_FOR_STOP,RFS_TRAFFIC_VIOLATION_TYPE,RFS_TRAFFIC_VIOLATION_CODE,RFS_RS_OFF_WITNESS,RFS_RS_MATCH_SUSPECT,RFS_RS_WITNESS_ID,RFS_RS_CARRY_SUS_OBJECT,RFS_RS_ACTIONS_INDICATIVE,RFS_RS_SUSPECT_LOOK,RFS_RS_DRUG_TRANS,RFS_RS_VIOLENT_CRIME,RFS_RS_REASON_SUSP,RFS_RS_MATCH_VEHICLE,RFS_RS_CODE,RFS_EC_DISCIPLINE_CODE,RFS_EC_DISCIPLINE,RFS_PC_OFF_WITNESS,RFS_PC_MATCH_SUSPECT,RFS_PC_WITNESS_ID,RFS_PC_CARRY_SUS_OBJECT,RFS_PC_ACTIONS_INDICATIVE,RFS_PC_SUSPECT_LOOK,RFS_PC_DRUG_TRANS,RFS_PC_VIOLENT_CRIME,RFS_PC_REASON_SUSP,RFS_PC_MATCH_VEHICLE,RFS_PC_CODE,RFS_RG_TRAFFIC_MOVING,RFS_RG_TRAFFIC_EQUIPMENT,RFS_RG_TRAFFIC_NON_MOVING,RFS_RG_OFF_WITNESS,RFS_RG_MATCH_SUSPECT,RFS_RG_MATCH_VEHICLE,RFS_RG_WITNESS_ID,RFS_RG_CARRY_SUS_OBJECT,RFS_RG_ACTIONS_INDICATIVE,RFS_RG_SUSPECT_LOOK,RFS_RG_DRUG_TRANS,RFS_RG_VIOLENT_CRIME,RFS_RG_REASON_SUSP,RFS_RG_PROBABLE_CAUSE,RFS_RG_WELFARE_AND_INST,RFS_RG_KNOWN_PAROLE,RFS_RG_OUTSTANDING_WARRANT,RFS_RG_TRUANT,RFS_RG_CONSENSUAL_SEARCH,RFS_RG_DISCIPLINE,RFS_RG_SCHOOL_POLICY,RFS_RG_NOT_COMMUNICATED,NFA_WRITTEN_STATEMENT,NFA_ASKED_SEARCH_PER,NFA_ASKED_SEARCH_PROP,NFA_ASKED_ID_PASSENGER,NFA_ASKED_PAROLE,NFA_CURB_DETENT,NFA_SOBRIETY_TEST,NFA_PATCAR_DETENT,NFA_CANINE_SEARCH,NFA_PHOTO,NFA_REMOVED_VEHICLE_ORDER,NFA_PROP_SEIZE,NFA_RAN_NAME_PASSENGER,NFA_SEARCH_PERSON,NFA_SEARCH_PROPERTY,NFA_TERRY_FRISK,NFA_VEHICLE_IMPOUND,NFA_NONE,NFA_SEARCH_PERS_CONSENT,NFA_SEARCH_PROP_CONSENT,OFA_HANDCUFFED,OFA_BATON_DRAWN,OFA_BATON_USED,OFA_CHEM_SPRAY,OFA_ELECT_DEVICE_POINT,OFA_ELECT_DEVICE_STUN,OFA_ELECT_DEVICE_DART,OFA_FIREARM_UNHOLSTERED,OFA_FIREARM_POINT,OFA_FIREARM_DISCHARGE,OFA_IMPACT_PROJECTILE_POINT,OFA_IMPACT_PROJECTILE_DISCHARGE,OFA_CANINE_COMPLIANCE,OFA_CANINE_BITE,OFA_REMOVED_VEHICLE_PHYCONTACT,OFA_PHYSICAL_COMPLIANCE,OFA_USE_VEHICLE,OFA_NONE,BFS_OFFICER_SAFETY,BFS_SEARCH_WARRANT,BFS_PAROLE,BFS_SUSPECT_WEAPON,BFS_VISIBLE_CONTRABAND,BFS_ODOR_CONTRABAND,BFS_CANINE_DETECT,BFS_EVIDENCE,BFS_INCIDENT,BFS_EXIGENT_CIRCUM,BFS_VEHICLE_INVENT,BFS_SCHOOL_POLICY,BFS_CONSENT_GIVEN,CTP_VERBAL,CTP_WRITTEN,CTP_IMPLIED,CED_NONE_CONTRABAND,CED_FIREARM,CED_AMMUNITION,CED_WEAPON,CED_DRUGS,CED_ALCOHOL,CED_MONEY,CED_DRUG_PARAPHERNALIA,CED_STOLEN_PROP,CED_ELECT_DEVICE,CED_OTHER_CONTRABAND,BPS_SAFEKEEPING,BPS_CONTRABAND,BPS_EVIDENCE,BPS_IMPOUND_VEHICLE,BPS_ABANDON_PROP,BPS_VIOLATE_SCHOOL,TPS_FIREARM,TPS_AMMUNITION,TPS_WEAPON,TPS_DRUGS,TPS_ALCOHOL,TPS_MONEY,TPS_DRUG_PARAPHERNALIA,TPS_STOLEN_PROP,TPS_CELLPHONE,TPS_VEHICLE,TPS_CONTRABAND,ROS_NO_ACTION,ROS_CITATION,ROS_IN_FIELD_CITE_RELEASE,ROS_CUSTODIAL_WARRANT,ROS_CUSTODIAL_WITHOUT_WARRANT,ROS_FIELD_INTERVIEW_CARD,ROS_NONCRIMINAL_TRANSPORT,ROS_CONTACT_LEGAL_GUARDIAN,ROS_PSYCH_HOLD,ROS_US_HOMELAND,ROS_REFERRAL_SCHOOL_ADMIN,ROS_REFERRAL_SCHOOL_COUNSELOR,ROS_WRITTEN_WARNING,ROS_VERBAL_WARNING,ROS_CITATION_CDS,ROS_IN_FIELD_CITE_RELEASE_CDS,ROS_CUSTODIAL_WOUT_WARRANT_CDS,ROS_VERBAL_WARNING_CDS,ROS_WRITTEN_WARNING_CDS,county_name
0,W389724007L72FIA0IWQ,1,CA0389700,UC-SAN FRANCISCO PD,0,10:25:00,2024-01-06,110,SAN FRANCISCO,NaN,NaN,0,0,2,1,1,50,7,0,0,1,0,0,0,0,0,1,0,2,NaN,NaN,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,32111.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,1.0000,1.0000,1.0000,0.0000,0.0000,NaN,NaN,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0,0,0,1,0,0,0,0,0,0,0,0,0,0

In [82]:
cali_df = reduce_memory_usage(cali_df, verbose=True)

Memory usage before: 9748.80 MB
Memory usage after: 5758.58 MB
Reduced by 40.9%


In [83]:
cali_df

,DOJ_RECORD_ID,PERSON_NUMBER,AGENCY_ORI,AGENCY_NAME,NON_REPORTING_AGENCY,TIME_OF_STOP,DATE_OF_STOP,STOP_DURATION,LOC_CLOSEST_CITY,SCHOOL_CODE,SCHOOL_NAME,STOP_STUDENT,K12_SCHOOL_GROUNDS,RAE_FULL,G_FULL,SOR_STRAIGHT,AGE,AGE_GROUP,LIMITED_ENGLISH_FLUENCY,PD_FULL,PERSON_UNHOUSED,PASSENGER_IN_VEHICLE,INSIDE_RESIDENCE,WELFARE_WELLNESS_CHECK,TOS_VEHICULAR,TOS_BICYCLE,TOS_PEDESTRIAN,CALL_FOR_SERVICE,REASON_FOR_STOP,RFS_TRAFFIC_VIOLATION_TYPE,RFS_TRAFFIC_VIOLATION_CODE,RFS_RS_OFF_WITNESS,RFS_RS_MATCH_SUSPECT,RFS_RS_WITNESS_ID,RFS_RS_CARRY_SUS_OBJECT,RFS_RS_ACTIONS_INDICATIVE,RFS_RS_SUSPECT_LOOK,RFS_RS_DRUG_TRANS,RFS_RS_VIOLENT_CRIME,RFS_RS_REASON_SUSP,RFS_RS_MATCH_VEHICLE,RFS_RS_CODE,RFS_EC_DISCIPLINE_CODE,RFS_EC_DISCIPLINE,RFS_PC_OFF_WITNESS,RFS_PC_MATCH_SUSPECT,RFS_PC_WITNESS_ID,RFS_PC_CARRY_SUS_OBJECT,RFS_PC_ACTIONS_INDICATIVE,RFS_PC_SUSPECT_LOOK,RFS_PC_DRUG_TRANS,RFS_PC_VIOLENT_CRIME,RFS_PC_REASON_SUSP,RFS_PC_MATCH_VEHICLE,RFS_PC_CODE,RFS_RG_TRAFFIC_MOVING,RFS_RG_TRAFFIC_EQUIPMENT,RFS_RG_TRAFFIC_NON_MOVING,RFS_RG_OFF_WITNESS,RFS_RG_MATCH_SUSPECT,RFS_RG_MATCH_VEHICLE,RFS_RG_WITNESS_ID,RFS_RG_CARRY_SUS_OBJECT,RFS_RG_ACTIONS_INDICATIVE,RFS_RG_SUSPECT_LOOK,RFS_RG_DRUG_TRANS,RFS_RG_VIOLENT_CRIME,RFS_RG_REASON_SUSP,RFS_RG_PROBABLE_CAUSE,RFS_RG_WELFARE_AND_INST,RFS_RG_KNOWN_PAROLE,RFS_RG_OUTSTANDING_WARRANT,RFS_RG_TRUANT,RFS_RG_CONSENSUAL_SEARCH,RFS_RG_DISCIPLINE,RFS_RG_SCHOOL_POLICY,RFS_RG_NOT_COMMUNICATED,NFA_WRITTEN_STATEMENT,NFA_ASKED_SEARCH_PER,NFA_ASKED_SEARCH_PROP,NFA_ASKED_ID_PASSENGER,NFA_ASKED_PAROLE,NFA_CURB_DETENT,NFA_SOBRIETY_TEST,NFA_PATCAR_DETENT,NFA_CANINE_SEARCH,NFA_PHOTO,NFA_REMOVED_VEHICLE_ORDER,NFA_PROP_SEIZE,NFA_RAN_NAME_PASSENGER,NFA_SEARCH_PERSON,NFA_SEARCH_PROPERTY,NFA_TERRY_FRISK,NFA_VEHICLE_IMPOUND,NFA_NONE,NFA_SEARCH_PERS_CONSENT,NFA_SEARCH_PROP_CONSENT,OFA_HANDCUFFED,OFA_BATON_DRAWN,OFA_BATON_USED,OFA_CHEM_SPRAY,OFA_ELECT_DEVICE_POINT,OFA_ELECT_DEVICE_STUN,OFA_ELECT_DEVICE_DART,OFA_FIREARM_UNHOLSTERED,OFA_FIREARM_POINT,OFA_FIREARM_DISCHARGE,OFA_IMPACT_PROJECTILE_POINT,OFA_IMPACT_PROJECTILE_DISCHARGE,OFA_CANINE_COMPLIANCE,OFA_CANINE_BITE,OFA_REMOVED_VEHICLE_PHYCONTACT,OFA_PHYSICAL_COMPLIANCE,OFA_USE_VEHICLE,OFA_NONE,BFS_OFFICER_SAFETY,BFS_SEARCH_WARRANT,BFS_PAROLE,BFS_SUSPECT_WEAPON,BFS_VISIBLE_CONTRABAND,BFS_ODOR_CONTRABAND,BFS_CANINE_DETECT,BFS_EVIDENCE,BFS_INCIDENT,BFS_EXIGENT_CIRCUM,BFS_VEHICLE_INVENT,BFS_SCHOOL_POLICY,BFS_CONSENT_GIVEN,CTP_VERBAL,CTP_WRITTEN,CTP_IMPLIED,CED_NONE_CONTRABAND,CED_FIREARM,CED_AMMUNITION,CED_WEAPON,CED_DRUGS,CED_ALCOHOL,CED_MONEY,CED_DRUG_PARAPHERNALIA,CED_STOLEN_PROP,CED_ELECT_DEVICE,CED_OTHER_CONTRABAND,BPS_SAFEKEEPING,BPS_CONTRABAND,BPS_EVIDENCE,BPS_IMPOUND_VEHICLE,BPS_ABANDON_PROP,BPS_VIOLATE_SCHOOL,TPS_FIREARM,TPS_AMMUNITION,TPS_WEAPON,TPS_DRUGS,TPS_ALCOHOL,TPS_MONEY,TPS_DRUG_PARAPHERNALIA,TPS_STOLEN_PROP,TPS_CELLPHONE,TPS_VEHICLE,TPS_CONTRABAND,ROS_NO_ACTION,ROS_CITATION,ROS_IN_FIELD_CITE_RELEASE,ROS_CUSTODIAL_WARRANT,ROS_CUSTODIAL_WITHOUT_WARRANT,ROS_FIELD_INTERVIEW_CARD,ROS_NONCRIMINAL_TRANSPORT,ROS_CONTACT_LEGAL_GUARDIAN,ROS_PSYCH_HOLD,ROS_US_HOMELAND,ROS_REFERRAL_SCHOOL_ADMIN,ROS_REFERRAL_SCHOOL_COUNSELOR,ROS_WRITTEN_WARNING,ROS_VERBAL_WARNING,ROS_CITATION_CDS,ROS_IN_FIELD_CITE_RELEASE_CDS,ROS_CUSTODIAL_WOUT_WARRANT_CDS,ROS_VERBAL_WARNING_CDS,ROS_WRITTEN_WARNING_CDS,county_name
0,W389724007L72FIA0IWQ,1,CA0389700,UC-SAN FRANCISCO PD,0,10:25:00,2024-01-06,110,SAN FRANCISCO,NaN,NaN,0,0,2,1,1,50,7,0,0,1,0,0,0,0,0,1,0,2,NaN,NaN,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,32111.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,1.0000,1.0000,1.0000,0.0000,0.0000,NaN,NaN,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0,0,0,1,0,0,0,0,0,0,0,0,0,0

# Concating DataFrames

## Cleaning Each DF

In [84]:
standard_cols = [
    "date", 
    "agency_name" ,
    "ori",
    "state",
    "location",
    "subject_race",
    "subject_age",
    "subject_gender" ,
    "force_used",
    "indication_of_injury" ,
    "incident_id"
]

### NJ

In [85]:
nj_df.head(1).T

,0
Form ID,866515
County,Ocean
Agency Name,Berkeley Twp PD
Officer Name,Michael Bulwinski
User ID,10818.0000
Incident ID,OCEAN-BERKELEY TWP PD-23-003647
Report Number,UOF23-11-52
Incident Case Number,23-003647
Incident Date,11/6/2023
Other Officer Involved,Not Provided


In [86]:
nj_df.columns

Index(['Form ID', 'County', 'Agency Name', 'Officer Name', 'User ID',
       'Incident ID', 'Report Number', 'Incident Case Number', 'Incident Date',
       'Other Officer Involved', 'Officer In Uniform', 'Incident Municipality',
       'Indoor Or Outdoor', 'Incident Weather', 'Video Footage', 'Video Type',
       'Incident Lighting', 'Location Type', 'Incident Type', 'Contact Origin',
       'Planned Contact', 'Officer Age', 'Officer Race/Ethnicity',
       'Officer Rank', 'Officer Gender', 'Officer Injury Type',
       'Officer Injuries Injured', 'Officer Medical Treatment',
       'Officer Hospital Treatment', 'Total Sub Injured In Incident',
       'Subject Injured In Incident', 'Subject Injured Prior To Incident',
       'Perceived Condition Of Subject', 'Subject Actions',
       'Subject Resistance', 'Subject Medical Treatment',
       'Subject Injury Type', 'Subject Arrested',
       'Reason Subject Not Arrested', 'Subject Type', 'Subject Age',
       'Subject Race/Ethnicity', '

In [87]:
nj_cols_to_keep = [
    'Form ID', 'County', 'Agency Name', 'Incident ID', 'Report Number', 'Incident Case Number', 'Incident Date', 'Incident Type', 
    'Subject Actions',
    'Subject Resistance', 'Subject Medical Treatment',
    'Subject Injury Type', 'Subject Arrested',
    'Reason Subject Not Arrested', 'Subject Type', 'Subject Age',
    'Subject Race/Ethnicity', 'Subject Gender', 'Force Type'
]
nj_df = nj_df[nj_cols_to_keep]

In [88]:
nj_df = nj_df.add_prefix("NJ_")

nj_df = (
    nj_df.assign(
        date = lambda x: x["NJ_Incident Date"],
        agency_name = lambda x: x["NJ_Agency Name"],
        ori = None,
        location = lambda x: x["NJ_County"],
        subject_race = lambda x: x["NJ_Subject Race/Ethnicity"],
        subject_age = lambda x: x["NJ_Subject Age"],
        subject_gender = lambda x: x["NJ_Subject Gender"],
        force_used = lambda x: x["NJ_Force Type"],
        indication_of_injury = None,
        incident_id = lambda x: x[["NJ_Incident ID", "NJ_Report Number", "NJ_Incident Case Number"]].apply(
                lambda row: [str(v) for v in row if pd.notna(v)],
                axis=1
            ))
)
nj_df.head()

,NJ_Form ID,NJ_County,NJ_Agency Name,NJ_Incident ID,NJ_Report Number,NJ_Incident Case Number,NJ_Incident Date,NJ_Incident Type,NJ_Subject Actions,NJ_Subject Resistance,NJ_Subject Medical Treatment,NJ_Subject Injury Type,NJ_Subject Arrested,NJ_Reason Subject Not Arrested,NJ_Subject Type,NJ_Subject Age,NJ_Subject Race/Ethnicity,NJ_Subject Gender,NJ_Force Type,date,agency_name,ori,location,subject_race,subject_age,subject_gender,force_used,indication_of_injury,incident_id
0,866515,Ocean,Berkeley Twp PD,OCEAN-BERKELEY TWP PD-23-003647,UOF23-11-52,23-003647,11/6/2023,Assault,Attempt to escape from Custody,Passive Resistor,NaN,NaN,TRUE,NaN,Person,28,Hispanic,Male,Used take down on,11/6/2023,Berkeley Twp PD,None,Ocean,Hispanic,28,Male,Used take down on,None,"[OCEAN-BERKELEY TWP PD-23-003647, UOF23-11-52,..."
1,91681,Ocean,Toms River Twp PD,OCEAN-TOMS RIVER TWP PD-21-20078,UOF21-5-48,21-20078,5/13/2021,"Welfare Check, Potential Mental Health Incident","Resisted arrest/police officer control, Verbal...","Verbal, Resistive tension (stiffening, tigheni...",NaN,NaN,FALSE,Medical/Mental Health Incident,Person,55,White,Female,Used arms,5/13/2021,Toms River Twp PD,None,Ocean,White,55,Female,Used arms,None,"[OCEAN-TOMS RIVER TWP PD-21-20078, UOF21-5-48,..."
2,50307,Cumberland,Vineland PD,CUMBERLAND-VINELAND PD-2021-4288,UOF21-1-19,2021-4288,1/27/2021,MV/Traffic Stop,"Resisted arrest/police officer control, Attack...","Resistive tension (stiffening, tighening muscl...","EMS on scene, Hospital",Abrasion/Laceration/Puncture,TRUE,NaN,Person,37,Hispanic,Male,Canine bit (apprehension),1/27/2021,Vineland PD,None,Cumberland,Hispanic,37,Male,Canine bit (apprehension),None,"[CUMBERLAND-VINELAND PD-2021-4288, UOF21-1-19,..."
3,201634,Ocean,Lakewood PD,OCEAN-LAKEWOOD PD-22-001779,UOF22-1-9,22-001779,1/8/2022,Medical Emergency,"Strike with open hand, fist, or elbow",Active Resistor,NaN,NaN,FALSE,Medical/Mental Health Incident,Person,35,Hispanic,Female,Used arms/hands,1/8/2022,Lakewood PD,None,Ocean,Hispanic,35,Female,Used arms/hands,None,"[OCEAN-LAKEWOOD PD-22-001779, UOF22-1-9, 22-00..."
4,1594362,Bergen,Rutherford PD,BERGEN-RUTHERFORD PD-25-08463,UOF25-5-1,25-08463,5/11/2025,Theft/Shoplifting,Attempt to escape from Custody,Active Resistor,NaN,NaN,TRUE,NaN,Person,19,Hispanic,Male,Used take down on,5/11/2025,Rutherford PD,None,Bergen,Hispanic,19,Male,Used take down on,None,"[BERGEN-RUTHERFORD PD-25-08463, UOF25-5-1, 25-..."


In [89]:
print(nj_df.shape)

(97877, 29)


### Ohio

In [90]:
ohio_df.head(1).T

,0
AgencyName,Akron
IncidentNumber,202400000028
IncidentDate,1/1/2024
InitialContactCircumstances,Unknown and is Unlikely to Ever be Known
LocationType,Unknown And Is Unlikely To Ever Be Known
SubjectResistanceTypes,Failing to Comply to Verbal Commands | Pulling...
SubjectArmed/BelievedToBeArmed,No
SubjectRaceEthnicity,White
SubjectGender,Male
SubjectInjuries,NaN


In [91]:
ohio_df = ohio_df.add_prefix("Ohio_")

ohio_df = (
    ohio_df.assign(
        date = lambda x: x["Ohio_IncidentDate"],
        agency_name = lambda x: x["Ohio_AgencyName"],
        ori = None,
        location = lambda x: x["Ohio_AgencyName"],
        subject_race = lambda x: x["Ohio_SubjectRaceEthnicity"],
        subject_age = None,
        subject_gender = lambda x: x["Ohio_SubjectGender"],
        force_used = lambda x: x["Ohio_OfficerResponse/ForceTypes"],
        indication_of_injury = lambda x: x["Ohio_SubjectInjuries"],
        incident_id = lambda x: x[["Ohio_IncidentNumber"]].apply(
                lambda row: [str(v) for v in row if pd.notna(v)],
                axis=1
            ))
)
ohio_df.head()

,Ohio_AgencyName,Ohio_IncidentNumber,Ohio_IncidentDate,Ohio_InitialContactCircumstances,Ohio_LocationType,Ohio_SubjectResistanceTypes,Ohio_SubjectArmed/BelievedToBeArmed,Ohio_SubjectRaceEthnicity,Ohio_SubjectGender,Ohio_SubjectInjuries,Ohio_SubjectImpairments,Ohio_OfficerType,Ohio_OfficerResponse/ForceTypes,Ohio_OfficerRaceEthnicity,Ohio_OfficerGender,Ohio_OfficerInjuries,date,agency_name,ori,location,subject_race,subject_age,subject_gender,force_used,indication_of_injury,incident_id
0,Akron,202400000028,1/1/2024,Unknown and is Unlikely to Ever be Known,Unknown And Is Unlikely To Ever Be Known,Failing to Comply to Verbal Commands | Pulling...,No,White,Male,NaN,NaN,Law Enforcement,Other Force Type Used | Restraining Hold | Oth...,White | White,Female | Male,Apparent Minor Injury | None,1/1/2024,Akron,None,Akron,White,None,Male,Other Force Type Used | Restraining Hold | Oth...,NaN,[202400000028]
1,Ravenna,2240010006,1/1/2024,Other Circumstances,Government Office,Verbally Threatening Others,No,White,Male,NaN,Alcohol Impairment,Law Enforcement,Restraining Hold,White | White,Male | Male,NaN,1/1/2024,Ravenna,None,Ravenna,White,None,Male,Restraining Hold,NaN,[2240010006]
2,Holland,24-000001-1,1/1/2024,Responding to Other Unlawful or Suspicious Act...,Single Family Home,Failing to Comply to Verbal Commands | Resisti...,No,Black or African American,Male,NaN,NaN,Law Enforcement,Balance Displacement | Electronic Control Devi...,White | White,Male | Male,NaN,1/1/2024,Holland,None,Holland,Black or African American,None,Male,Balance Displacement | Electronic Control Devi...,NaN,[24-000001-1]
3,Holland,24-000001-1,1/1/2024,Responding to Other Unlawful or Suspicious Act...,Single Family Home,Failing to Comply to Verbal Commands | Resisti...,No,Black or African American,Male,NaN,NaN,Law Enforcement,Balance Displacement | Electronic Control Devi...,White | White,Male | Male,NaN,1/1/2024,Holland,None,Holland,Black or African American,None,Male,Balance Displacement | Electronic Control Devi...,NaN,[24-000001-1]
4,Woodlawn,202400000001,1/1/2024,Traffic Stop,Single Family Home,Firearm Displayed at an Officer or Another,Yes,Black or African American,Female,NaN,NaN,Law Enforcement,Other Force Type Used,Black or African American | White,Male | Male,NaN,1/1/2024,Woodlawn,None,Woodlawn,Black or African American,None,Female,Other Force Type Used,NaN,[202400000001]


In [92]:
ohio_df

,Ohio_AgencyName,Ohio_IncidentNumber,Ohio_IncidentDate,Ohio_InitialContactCircumstances,Ohio_LocationType,Ohio_SubjectResistanceTypes,Ohio_SubjectArmed/BelievedToBeArmed,Ohio_SubjectRaceEthnicity,Ohio_SubjectGender,Ohio_SubjectInjuries,Ohio_SubjectImpairments,Ohio_OfficerType,Ohio_OfficerResponse/ForceTypes,Ohio_OfficerRaceEthnicity,Ohio_OfficerGender,Ohio_OfficerInjuries,date,agency_name,ori,location,subject_race,subject_age,subject_gender,force_used,indication_of_injury,incident_id
0,Akron,202400000028,1/1/2024,Unknown and is Unlikely to Ever be Known,Unknown And Is Unlikely To Ever Be Known,Failing to Comply to Verbal Commands | Pulling...,No,White,Male,NaN,NaN,Law Enforcement,Other Force Type Used | Restraining Hold | Oth...,White | White,Female | Male,Apparent Minor Injury | None,1/1/2024,Akron,None,Akron,White,None,Male,Other Force Type Used | Restraining Hold | Oth...,NaN,[202400000028]
1,Ravenna,2240010006,1/1/2024,Other Circumstances,Government Office,Verbally Threatening Others,No,White,Male,NaN,Alcohol Impairment,Law Enforcement,Restraining Hold,White | White,Male | Male,NaN,1/1/2024,Ravenna,None,Ravenna,White,None,Male,Restraining Hold,NaN,[2240010006]
2,Holland,24-000001-1,1/1/2024,Responding to Other Unlawful or Suspicious Act...,Single Family Home,Failing to Comply to Verbal Commands | Resisti...,No,Black or African American,Male,NaN,NaN,Law Enforcement,Balance Displacement | Electronic Control Devi...,White | White,Male | Male,NaN,1/1/2024,Holland,None,Holland,Black or African American,None,Male,Balance Displacement | Electronic Control Devi...,NaN,[24-000001-1]
3,Holland,24-000001-1,1/1/2024,Responding to Other Unlawful or Suspicious Act...,Single Family Home,Failing to Comply to Verbal Commands | Resisti...,No,Black or African American,Male,NaN,NaN,Law Enforcement,Balance Displacement | Electronic Control Devi...,White | White,Male | Male,NaN,1/1/2024,Holland,None,Holland,Black or African American,None,Male,Balance Displacement | Electronic Control Devi...,NaN,[24-000001-1]
4,Woodlawn,202400000001,1/1/2024,Traffic Stop,Single Family Home,Firearm Displayed at an Officer or Another,Yes,Black or African American,Female,NaN,NaN,Law Enforcement,Other Force Type Used,Black or African American | White,Male | Male,NaN,1/1/2024,Woodlawn,None,Woodlawn,Black or African American,None,Female,Other Force Type Used,NaN,[202400000001]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3253,Wooster,24-75026,9/9/2024,Responding to Other Unlawful or Suspicious Act...,Street,Failing to Comply to Verbal Commands | Resisti...,No,White,Male,NaN,Drug Impairment | Mental Health Condition,Law Enforcement,Restraining Hold | Take Down,White,Male,NaN,9/9/2024,Wooster,None,Wooster,White,None,Male,Restraining Hold | Take Down,NaN,[24-75026]
3254,Liberty Township,24-0009655,9/9/2024,Routine Patrol/On-view (Other than Traffic),Parking Lot,Attempt to Escape/Flee From Custody | Failing ...,Yes,Black or African American,Male,NaN,NaN,Law Enforcement,Restraining Hold | Take Down,Unknown/Not Reported,Pending Further Investigation,NaN,9/9/2024,Liberty Township,None,Liberty Township,Black or African American,None,Male,Restraining Hold | Take Down,NaN,[24-0009655]
3255,Cleveland,2024-265997,9/9/2024,Unknown and is Unlikely to Ever be Known,Unknown And Is Unlikely To Ever Be Known,Pulling Away | Resisting Being Handcuffed or A...,Unknown and is Unlikely to Ever be Known,Black or African American,Female,NaN,NaN,Law Enforcement,Balance Displacement | Restraining Hold,Black or African American | White,Male | Female,NaN,9/9/2024,Cleveland,None,Cleveland,Black or African American,None,Female,Balance Displacement | Restraining Hold,NaN,[2024-265997]
3256,Zanesville,24-020795,9/9/2024,Responding to Other Unlawful or Suspicious Act...,Single Family Home,Attempt to Escape/Flee From Custody | Deadweig...,No,White,Male,Apparent Minor Injury,NaN,Law Enforcement,Balance Displacement | Restraining Hold | Pres

### NY

In [93]:
ny_df.head(1).T

,0
TRI Incident Number,2024032962162
ForceType,Physical Force
Occurrence Date,08/31/2024
Incident Pct,13
Patrol Borough,PBMS
YearMonthShort,2024 Aug
BasisForEncounter,CRIME/VIOLATION IN PROGRESS
Subject Injury Level,No Injury
Subject Injured,N
Age,17.0000


In [94]:
ny_df = ny_df.add_prefix("NY_")


In [95]:
ny_df = (
    ny_df.assign(
        date = lambda x: x["NY_Occurrence Date"],
        agency_name = lambda x: x["NY_Incident Pct"],
        ori = None,
        location = lambda x: x["NY_Patrol Borough"],
        subject_race = lambda x: x["NY_Subject Race"],
        subject_age = lambda x: x["NY_Age"],
        subject_gender = lambda x: x["NY_Subject Gender"],
        force_used = lambda x: x["NY_ForceType"],
        indication_of_injury = lambda x: x["NY_Subject Injured"],
        incident_id = lambda x: x[["NY_TRI Incident Number"]].apply(
                lambda row: [str(v) for v in row if pd.notna(v)],
                axis=1
            ))
)
ny_df.head()

,NY_TRI Incident Number,NY_ForceType,NY_Occurrence Date,NY_Incident Pct,NY_Patrol Borough,NY_YearMonthShort,NY_BasisForEncounter,NY_Subject Injury Level,NY_Subject Injured,NY_Age,NY_Subject Race,NY_Subject Gender,NY_Force Against MOS,NY_Subject Used Force,date,agency_name,ori,location,subject_race,subject_age,subject_gender,force_used,indication_of_injury,incident_id
0,2024032962162,Physical Force,08/31/2024,13,PBMS,2024 Aug,CRIME/VIOLATION IN PROGRESS,No Injury,N,17.0000,BLACK,MALE,Physical Force,Y,08/31/2024,13,None,PBMS,BLACK,17.0000,MALE,Physical Force,N,[2024032962162]
1,2024033962356,Physical Force,08/31/2024,14,PBMS,2024 Aug,CRIME/VIOLATION IN PROGRESS,No Injury,N,30.0000,BLACK,MALE,Physical Force,Y,08/31/2024,14,None,PBMS,BLACK,30.0000,MALE,Physical Force,N,[2024033962356]
2,2024090962120,Physical Force,08/31/2024,71,PBBS,2024 Aug,CRIME/VIOLATION IN PROGRESS,No Injury,N,19.0000,OTHER,MALE,Physical Force,Y,08/31/2024,71,None,PBBS,OTHER,19.0000,MALE,Physical Force,N,[2024090962120]
3,2024096962122,Physical Force,08/31/2024,77,PBBN,2024 Aug,CRIME/VIOLATION IN PROGRESS,No Injury,N,38.0000,BLACK,MALE,Physical Force,Y,08/31/2024,77,None,PBBN,BLACK,38.0000,MALE,Physical Force,N,[2024096962122]
4,2024081962083,Physical Force,08/31/2024,62,PBBS,2024 Aug,CRIME/VIOLATION IN PROGRESS,Physical Injury,Y,17.0000,HISPANIC,MALE,Physical Force,Y,08/31/2024,62,None,PBBS,HISPANIC,17.0000,MALE,Physical Force,Y,[2024081962083]


In [96]:
ny_df["force_used"].isna().sum()

np.int64(0)

In [97]:
ny_uof_long = ny_uof_long.add_prefix("NY_")


In [98]:
ny_uof_long["NY_subject_ethnicity"] = ny_uof_long["NY_subject_ethnicity"].fillna(ny_uof_long["NY_subject_race"])

In [99]:
ny_uof_long["NY_subject_ethnicity"].info()

<class 'pandas.Series'>
Index: 42881 entries, 0 to 870900
Series name: NY_subject_ethnicity
Non-Null Count  Dtype
--------------  -----
42881 non-null  str  
dtypes: str(1)
memory usage: 670.0 KB


In [100]:
ny_uof_long.head(1).T

,0
NY_incident_id,0
NY_Response ID\n(5/16/25 File),1
NY_Agency Name,Akron Vg PD
NY_Incident Date,2020-12-22 00:00:00
NY_City/town/village (location Where Incident Occurred),Akron
NY_County (location where incident occurred),Erie
NY_Circumstance,Response to unlawful or suspicious activity
NY_Officer 1 - Use of Force Type,Pointed Firearm
NY_Officer 1 Age,27.0000
NY_Officer 1 Sex,Male


In [101]:
ny_long_force_cols = [c for c in ny_uof_long.columns if "Use of Force Type" in c]
ny_long_force_cols

['NY_Officer 1 - Use of Force Type',
 'NY_Officer 2 - Use of Force Type',
 'NY_Officer 3 - Use of Force Type',
 'NY_Officer 4 - Use of Force Type',
 'NY_Officer 5 - Use of Force Type',
 'NY_Officer 6 - Use of Force Type',
 'NY_Officer 7 - Use of Force Type',
 'NY_Officer 8 - Use of Force Type',
 'NY_Officer 9 - Use of Force Type',
 'NY_Officer 10 - Use of Force Type',
 'NY_Officer 11 - Use of Force Type',
 'NY_Officer 12 - Use of Force Type',
 'NY_Officer 13 - Use of Force Type',
 'NY_Officer 14 - Use of Force Type',
 'NY_Officer 15 - Use of Force Type',
 'NY_Officer 16 - Use of Force Type']

In [102]:
ny_uof_long.shape

(42881, 92)

In [103]:
(
    ny_uof_long.assign(
        date = lambda x: x["NY_Incident Date"],
        agency_name = lambda x: x["NY_Agency Name"],
        ori = None,
        location = lambda x: x["NY_City/town/village (location Where Incident Occurred)"] + ", " + x["NY_County (location where incident occurred)"],
        subject_race = lambda x: x["NY_subject_race"],
        subject_age = lambda x: x["NY_subject_age"],
        subject_gender = lambda x: x["NY_subject_sex"],
        force_used = lambda x: x[ny_long_force_cols].apply(
            lambda row: ' | '.join([
                str(row[col]) for col in ny_long_force_cols 
                if (not pd.isna(row[col])) and (str(row[col]).strip() != "") and (str(row[col]).lower() != "nan" and str(row[col]) != "0")
            ]), axis=1
        ),
        indication_of_injury = None,
        incident_id = lambda x: x[["NY_Response ID\n(5/16/25 File)"]]
    )
)

,NY_incident_id,NY_Response ID\n(5/16/25 File),NY_Agency Name,NY_Incident Date,NY_City/town/village (location Where Incident Occurred),NY_County (location where incident occurred),NY_Circumstance,NY_Officer 1 - Use of Force Type,NY_Officer 1 Age,NY_Officer 1 Sex,NY_Officer 1 Race,NY_Officer 1 Ethnicity,NY_Officer 2 - Use of Force Type,NY_Officer 2 Age,NY_Officer 2 Sex,NY_Officer 2 Race,NY_Officer 2 Ethnicity,NY_Officer 3 - Use of Force Type,NY_Officer 3 Age,NY_Officer 3 Sex,NY_Officer 3 Race,NY_Officer 3 Ethnicity,NY_Officer 4 - Use of Force Type,NY_Officer 4 Age,NY_Officer 4 Sex,NY_Officer 4 Race,NY_Officer 4 Ethnicity,NY_Officer 5 - Use of Force Type,NY_Officer 5 Age,NY_Officer 5 Sex,NY_Officer 5 Race,NY_Officer 5 Ethnicity,NY_Officer 6 - Use of Force Type,NY_Officer 6 Age,NY_Officer 6 Sex,NY_Officer 6 Race,NY_Officer 6 Ethnicity,NY_Officer 7 - Use of Force Type,NY_Officer 7 Age,NY_Officer 7 Sex,NY_Officer 7 Race,NY_Officer 7 Ethnicity,NY_Officer 8 - Use of Force Type,NY_Officer 8 Age,NY_Officer 8 Sex,NY_Officer 8 Race,NY_Officer 8 Ethnicity,NY_Officer 9 - Use of Force Type,NY_Officer 9 Age,NY_Officer 9 Sex,NY_Officer 9 Race,NY_Officer 9 Ethnicity,NY_Officer 10 - Use of Force Type,NY_Officer 10 Age,NY_Officer 10 Sex,NY_Officer 10 Race,NY_Officer 10 Ethnicity,NY_Officer 11 - Use of Force Type,NY_Officer 11 Age,NY_Officer 11 Sex,NY_Officer 11 Race,NY_Officer 11 Ethnicity,NY_Officer 12 - Use of Force Type,NY_Officer 12 Age,NY_Officer 12 Sex,NY_Officer 12 Race,NY_Officer 12 Ethnicity,NY_Officer 13 - Use of Force Type,NY_Officer 13 Age,NY_Officer 13 Sex,NY_Officer 13 Race,NY_Officer 13 Ethnicity,NY_Officer 14 - Use of Force Type,NY_Officer 14 Age,NY_Officer 14 Sex,NY_Officer 14 Race,NY_Officer 14 Ethnicity,NY_Officer 15 - Use of Force Type,NY_Officer 15 Age,NY_Officer 15 Sex,NY_Officer 15 Race,NY_Officer 15 Ethnicity,NY_Officer 16 - Use of Force Type,NY_Officer 16 Age,NY_Officer 16 Sex,NY_Officer 16 Race,NY_Officer 16 Ethnicity,NY_subject_number,NY_subject_race,NY_subject_ethnicity,NY_subject_age,NY_subject_sex,date,agency_name,ori,location,subject_race,subject_age,subject_gender,force_used,indication_of_injury,incident_id
0,0,1,Akron Vg PD,2020-12-22,Akron,Erie,Response to unlawful or suspicious activity,Pointed Firearm,27.0000,Male,White,Non-Hispanic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,White,Non-Hispanic,49.0000,Male,2020-12-22,Akron Vg PD,None,"Akron, Erie",White,49.0000,Male,Pointed Firearm,None,1
25,1,2,Albany City PD,2020-11-10,Albany,Albany,Other,Used/Deployed Chemical Agent,37.0000,Male,White,Non-Hispanic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Black,Unknown,NaN,Male,2020-11-10,Albany City PD,None,"Albany, Albany",Black,NaN,Male,Used/Deployed Chemical Agent,None,2
50,2,3,Albany City PD,2020-11-20,Albany,Albany,Other,Pointed Firearm,45.0000,Male,White,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Black,Unknown,49.0000,Male,2020-11-20,Albany City PD,None,"Albany, Albany",Black,49.0000,Male,Pointed Firearm,None,3
75,3,4,Albany City PD,2020-12-08,Albany,Albany,Executing Arrest,Pointed Electronic Control Weapon,50.0000,Male,White,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Na

In [104]:
ny_uof_long = (
    ny_uof_long.assign(
        date = lambda x: x["NY_Incident Date"],
        agency_name = lambda x: x["NY_Agency Name"],
        ori = None,
        location = lambda x: x["NY_City/town/village (location Where Incident Occurred)"] + ", " + x["NY_County (location where incident occurred)"],
        subject_race = lambda x: x["NY_subject_race"],
        subject_age = lambda x: x["NY_subject_age"],
        subject_gender = lambda x: x["NY_subject_sex"],
        force_used = lambda x: x[ny_long_force_cols].apply(
            lambda row: ' | '.join([
                str(row[col]) for col in ny_long_force_cols 
                if (not pd.isna(row[col])) and (str(row[col]).strip() != "") and (str(row[col]).lower() != "nan" and str(row[col]) != "0")
            ]), axis=1
        ),
        indication_of_injury = None,
        incident_id = lambda x: x[["NY_Response ID\n(5/16/25 File)"]]
    )
)
ny_uof_long.head()

,NY_incident_id,NY_Response ID\n(5/16/25 File),NY_Agency Name,NY_Incident Date,NY_City/town/village (location Where Incident Occurred),NY_County (location where incident occurred),NY_Circumstance,NY_Officer 1 - Use of Force Type,NY_Officer 1 Age,NY_Officer 1 Sex,NY_Officer 1 Race,NY_Officer 1 Ethnicity,NY_Officer 2 - Use of Force Type,NY_Officer 2 Age,NY_Officer 2 Sex,NY_Officer 2 Race,NY_Officer 2 Ethnicity,NY_Officer 3 - Use of Force Type,NY_Officer 3 Age,NY_Officer 3 Sex,NY_Officer 3 Race,NY_Officer 3 Ethnicity,NY_Officer 4 - Use of Force Type,NY_Officer 4 Age,NY_Officer 4 Sex,NY_Officer 4 Race,NY_Officer 4 Ethnicity,NY_Officer 5 - Use of Force Type,NY_Officer 5 Age,NY_Officer 5 Sex,NY_Officer 5 Race,NY_Officer 5 Ethnicity,NY_Officer 6 - Use of Force Type,NY_Officer 6 Age,NY_Officer 6 Sex,NY_Officer 6 Race,NY_Officer 6 Ethnicity,NY_Officer 7 - Use of Force Type,NY_Officer 7 Age,NY_Officer 7 Sex,NY_Officer 7 Race,NY_Officer 7 Ethnicity,NY_Officer 8 - Use of Force Type,NY_Officer 8 Age,NY_Officer 8 Sex,NY_Officer 8 Race,NY_Officer 8 Ethnicity,NY_Officer 9 - Use of Force Type,NY_Officer 9 Age,NY_Officer 9 Sex,NY_Officer 9 Race,NY_Officer 9 Ethnicity,NY_Officer 10 - Use of Force Type,NY_Officer 10 Age,NY_Officer 10 Sex,NY_Officer 10 Race,NY_Officer 10 Ethnicity,NY_Officer 11 - Use of Force Type,NY_Officer 11 Age,NY_Officer 11 Sex,NY_Officer 11 Race,NY_Officer 11 Ethnicity,NY_Officer 12 - Use of Force Type,NY_Officer 12 Age,NY_Officer 12 Sex,NY_Officer 12 Race,NY_Officer 12 Ethnicity,NY_Officer 13 - Use of Force Type,NY_Officer 13 Age,NY_Officer 13 Sex,NY_Officer 13 Race,NY_Officer 13 Ethnicity,NY_Officer 14 - Use of Force Type,NY_Officer 14 Age,NY_Officer 14 Sex,NY_Officer 14 Race,NY_Officer 14 Ethnicity,NY_Officer 15 - Use of Force Type,NY_Officer 15 Age,NY_Officer 15 Sex,NY_Officer 15 Race,NY_Officer 15 Ethnicity,NY_Officer 16 - Use of Force Type,NY_Officer 16 Age,NY_Officer 16 Sex,NY_Officer 16 Race,NY_Officer 16 Ethnicity,NY_subject_number,NY_subject_race,NY_subject_ethnicity,NY_subject_age,NY_subject_sex,date,agency_name,ori,location,subject_race,subject_age,subject_gender,force_used,indication_of_injury,incident_id
0,0,1,Akron Vg PD,2020-12-22,Akron,Erie,Response to unlawful or suspicious activity,Pointed Firearm,27.0000,Male,White,Non-Hispanic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,White,Non-Hispanic,49.0000,Male,2020-12-22,Akron Vg PD,None,"Akron, Erie",White,49.0000,Male,Pointed Firearm,None,1
25,1,2,Albany City PD,2020-11-10,Albany,Albany,Other,Used/Deployed Chemical Agent,37.0000,Male,White,Non-Hispanic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Black,Unknown,NaN,Male,2020-11-10,Albany City PD,None,"Albany, Albany",Black,NaN,Male,Used/Deployed Chemical Agent,None,2
50,2,3,Albany City PD,2020-11-20,Albany,Albany,Other,Pointed Firearm,45.0000,Male,White,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Black,Unknown,49.0000,Male,2020-11-20,Albany City PD,None,"Albany, Albany",Black,49.0000,Male,Pointed Firearm,None,3
75,3,4,Albany City PD,2020-12-08,Albany,Albany,Executing Arrest,Pointed Electronic Control Weapon,50.0000,Male,White,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Na

### Cali

In [105]:
cali_df.head(1).T

,0
DOJ_RECORD_ID,W389724007L72FIA0IWQ
PERSON_NUMBER,1
AGENCY_ORI,CA0389700
AGENCY_NAME,UC-SAN FRANCISCO PD
NON_REPORTING_AGENCY,0
TIME_OF_STOP,10:25:00
DATE_OF_STOP,2024-01-06
STOP_DURATION,110
LOC_CLOSEST_CITY,SAN FRANCISCO
SCHOOL_CODE,NaN


In [106]:
(cali_df[["OFA_FIREARM_UNHOLSTERED", "OFA_FIREARM_POINT", "OFA_FIREARM_DISCHARGE"]] != 0).any(axis=1).sum()


np.int64(31444)

In [107]:
(cali_df[["OFA_FIREARM_DISCHARGE"]] != 0).any(axis=1).sum()


np.int64(1401)

In [108]:
cali_force_cols = [
                    c for c in cali_df.columns if "OFA_" in c 
                                        and (c != "OFA_NONE") 
                                        and (c != "OFA_HANDCUFFED")
                                        and (c != "OFA_BATON_DRAWN")
                                        and (c != "OFA_ELECT_DEVICE_POINT")
                                        and (c != "OFA_FIREARM_UNHOLSTERED")
                                        and (c != "OFA_FIREARM_POINT")
                                        and (c != "OFA_IMPACT_PROJECTILE_POINT")
                                        and (c != "OFA_CANINE_COMPLIANCE")
                  ]
cali_force_cols

['OFA_BATON_USED',
 'OFA_CHEM_SPRAY',
 'OFA_ELECT_DEVICE_STUN',
 'OFA_ELECT_DEVICE_DART',
 'OFA_FIREARM_DISCHARGE',
 'OFA_IMPACT_PROJECTILE_DISCHARGE',
 'OFA_CANINE_BITE',
 'OFA_REMOVED_VEHICLE_PHYCONTACT',
 'OFA_PHYSICAL_COMPLIANCE',
 'OFA_USE_VEHICLE']

In [109]:
cali_df[cali_force_cols].info()

<class 'pandas.DataFrame'>
RangeIndex: 5065428 entries, 0 to 5065427
Data columns (total 10 columns):
 #   Column                           Dtype  
---  ------                           -----  
 0   OFA_BATON_USED                   float32
 1   OFA_CHEM_SPRAY                   float32
 2   OFA_ELECT_DEVICE_STUN            float32
 3   OFA_ELECT_DEVICE_DART            float32
 4   OFA_FIREARM_DISCHARGE            float32
 5   OFA_IMPACT_PROJECTILE_DISCHARGE  float32
 6   OFA_CANINE_BITE                  float32
 7   OFA_REMOVED_VEHICLE_PHYCONTACT   float32
 8   OFA_PHYSICAL_COMPLIANCE          float32
 9   OFA_USE_VEHICLE                  float32
dtypes: float32(10)
memory usage: 193.2 MB


In [110]:
cali_df[cali_force_cols] = cali_df[cali_force_cols].fillna(0)

In [111]:
cali_df[cali_force_cols] = cali_df[cali_force_cols].astype(int)

In [112]:
cali_force_df = (
    cali_df[
        (cali_df[cali_force_cols] != 0).any(axis=1)
    ]
)
print(cali_force_df.shape)
cali_force_df.head()

(36375, 179)


,DOJ_RECORD_ID,PERSON_NUMBER,AGENCY_ORI,AGENCY_NAME,NON_REPORTING_AGENCY,TIME_OF_STOP,DATE_OF_STOP,STOP_DURATION,LOC_CLOSEST_CITY,SCHOOL_CODE,SCHOOL_NAME,STOP_STUDENT,K12_SCHOOL_GROUNDS,RAE_FULL,G_FULL,SOR_STRAIGHT,AGE,AGE_GROUP,LIMITED_ENGLISH_FLUENCY,PD_FULL,PERSON_UNHOUSED,PASSENGER_IN_VEHICLE,INSIDE_RESIDENCE,WELFARE_WELLNESS_CHECK,TOS_VEHICULAR,TOS_BICYCLE,TOS_PEDESTRIAN,CALL_FOR_SERVICE,REASON_FOR_STOP,RFS_TRAFFIC_VIOLATION_TYPE,RFS_TRAFFIC_VIOLATION_CODE,RFS_RS_OFF_WITNESS,RFS_RS_MATCH_SUSPECT,RFS_RS_WITNESS_ID,RFS_RS_CARRY_SUS_OBJECT,RFS_RS_ACTIONS_INDICATIVE,RFS_RS_SUSPECT_LOOK,RFS_RS_DRUG_TRANS,RFS_RS_VIOLENT_CRIME,RFS_RS_REASON_SUSP,RFS_RS_MATCH_VEHICLE,RFS_RS_CODE,RFS_EC_DISCIPLINE_CODE,RFS_EC_DISCIPLINE,RFS_PC_OFF_WITNESS,RFS_PC_MATCH_SUSPECT,RFS_PC_WITNESS_ID,RFS_PC_CARRY_SUS_OBJECT,RFS_PC_ACTIONS_INDICATIVE,RFS_PC_SUSPECT_LOOK,RFS_PC_DRUG_TRANS,RFS_PC_VIOLENT_CRIME,RFS_PC_REASON_SUSP,RFS_PC_MATCH_VEHICLE,RFS_PC_CODE,RFS_RG_TRAFFIC_MOVING,RFS_RG_TRAFFIC_EQUIPMENT,RFS_RG_TRAFFIC_NON_MOVING,RFS_RG_OFF_WITNESS,RFS_RG_MATCH_SUSPECT,RFS_RG_MATCH_VEHICLE,RFS_RG_WITNESS_ID,RFS_RG_CARRY_SUS_OBJECT,RFS_RG_ACTIONS_INDICATIVE,RFS_RG_SUSPECT_LOOK,RFS_RG_DRUG_TRANS,RFS_RG_VIOLENT_CRIME,RFS_RG_REASON_SUSP,RFS_RG_PROBABLE_CAUSE,RFS_RG_WELFARE_AND_INST,RFS_RG_KNOWN_PAROLE,RFS_RG_OUTSTANDING_WARRANT,RFS_RG_TRUANT,RFS_RG_CONSENSUAL_SEARCH,RFS_RG_DISCIPLINE,RFS_RG_SCHOOL_POLICY,RFS_RG_NOT_COMMUNICATED,NFA_WRITTEN_STATEMENT,NFA_ASKED_SEARCH_PER,NFA_ASKED_SEARCH_PROP,NFA_ASKED_ID_PASSENGER,NFA_ASKED_PAROLE,NFA_CURB_DETENT,NFA_SOBRIETY_TEST,NFA_PATCAR_DETENT,NFA_CANINE_SEARCH,NFA_PHOTO,NFA_REMOVED_VEHICLE_ORDER,NFA_PROP_SEIZE,NFA_RAN_NAME_PASSENGER,NFA_SEARCH_PERSON,NFA_SEARCH_PROPERTY,NFA_TERRY_FRISK,NFA_VEHICLE_IMPOUND,NFA_NONE,NFA_SEARCH_PERS_CONSENT,NFA_SEARCH_PROP_CONSENT,OFA_HANDCUFFED,OFA_BATON_DRAWN,OFA_BATON_USED,OFA_CHEM_SPRAY,OFA_ELECT_DEVICE_POINT,OFA_ELECT_DEVICE_STUN,OFA_ELECT_DEVICE_DART,OFA_FIREARM_UNHOLSTERED,OFA_FIREARM_POINT,OFA_FIREARM_DISCHARGE,OFA_IMPACT_PROJECTILE_POINT,OFA_IMPACT_PROJECTILE_DISCHARGE,OFA_CANINE_COMPLIANCE,OFA_CANINE_BITE,OFA_REMOVED_VEHICLE_PHYCONTACT,OFA_PHYSICAL_COMPLIANCE,OFA_USE_VEHICLE,OFA_NONE,BFS_OFFICER_SAFETY,BFS_SEARCH_WARRANT,BFS_PAROLE,BFS_SUSPECT_WEAPON,BFS_VISIBLE_CONTRABAND,BFS_ODOR_CONTRABAND,BFS_CANINE_DETECT,BFS_EVIDENCE,BFS_INCIDENT,BFS_EXIGENT_CIRCUM,BFS_VEHICLE_INVENT,BFS_SCHOOL_POLICY,BFS_CONSENT_GIVEN,CTP_VERBAL,CTP_WRITTEN,CTP_IMPLIED,CED_NONE_CONTRABAND,CED_FIREARM,CED_AMMUNITION,CED_WEAPON,CED_DRUGS,CED_ALCOHOL,CED_MONEY,CED_DRUG_PARAPHERNALIA,CED_STOLEN_PROP,CED_ELECT_DEVICE,CED_OTHER_CONTRABAND,BPS_SAFEKEEPING,BPS_CONTRABAND,BPS_EVIDENCE,BPS_IMPOUND_VEHICLE,BPS_ABANDON_PROP,BPS_VIOLATE_SCHOOL,TPS_FIREARM,TPS_AMMUNITION,TPS_WEAPON,TPS_DRUGS,TPS_ALCOHOL,TPS_MONEY,TPS_DRUG_PARAPHERNALIA,TPS_STOLEN_PROP,TPS_CELLPHONE,TPS_VEHICLE,TPS_CONTRABAND,ROS_NO_ACTION,ROS_CITATION,ROS_IN_FIELD_CITE_RELEASE,ROS_CUSTODIAL_WARRANT,ROS_CUSTODIAL_WITHOUT_WARRANT,ROS_FIELD_INTERVIEW_CARD,ROS_NONCRIMINAL_TRANSPORT,ROS_CONTACT_LEGAL_GUARDIAN,ROS_PSYCH_HOLD,ROS_US_HOMELAND,ROS_REFERRAL_SCHOOL_ADMIN,ROS_REFERRAL_SCHOOL_COUNSELOR,ROS_WRITTEN_WARNING,ROS_VERBAL_WARNING,ROS_CITATION_CDS,ROS_IN_FIELD_CITE_RELEASE_CDS,ROS_CUSTODIAL_WOUT_WARRANT_CDS,ROS_VERBAL_WARNING_CDS,ROS_WRITTEN_WARNING_CDS,county_name
51,U3800240116626931439,1,CA0380000,SAN FRANCISCO CO SO,0,17:02:00,2024-01-11,20,SAN FRANCISCO,NaN,NaN,0,0,2,1,1,35,6,0,0,0,0,0,0,0,0,1,0,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN,1.0000,0.0000,0,0,0.0000,0,0,0.0000,0.0000,0,0.0000,0,0.0000,0,1,1,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0,0,0,1,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,0.0000,0.00

In [113]:
cali_cols_to_keep = [
    "DOJ_RECORD_ID",
    "AGENCY_ORI", "AGENCY_NAME", 
    "DATE_OF_STOP",
    "county_name", "LOC_CLOSEST_CITY", 
    "RAE_FULL", "G_FULL", "SOR_STRAIGHT", "PD_FULL", "LIMITED_ENGLISH_FLUENCY", "AGE", 
    "REASON_FOR_STOP",
    "OFA_NONE"
]
cali_cols_to_keep += cali_force_cols

In [114]:
cali_force_df = cali_force_df[cali_cols_to_keep]

In [115]:
cali_force_df.head()

,DOJ_RECORD_ID,AGENCY_ORI,AGENCY_NAME,DATE_OF_STOP,county_name,LOC_CLOSEST_CITY,RAE_FULL,G_FULL,SOR_STRAIGHT,PD_FULL,LIMITED_ENGLISH_FLUENCY,AGE,REASON_FOR_STOP,OFA_NONE,OFA_BATON_USED,OFA_CHEM_SPRAY,OFA_ELECT_DEVICE_STUN,OFA_ELECT_DEVICE_DART,OFA_FIREARM_DISCHARGE,OFA_IMPACT_PROJECTILE_DISCHARGE,OFA_CANINE_BITE,OFA_REMOVED_VEHICLE_PHYCONTACT,OFA_PHYSICAL_COMPLIANCE,OFA_USE_VEHICLE
51,U3800240116626931439,CA0380000,SAN FRANCISCO CO SO,2024-01-11,San Francisco,SAN FRANCISCO,2,1,1,0,0,35,4,0.0000,0,0,0,0,0,0,0,1,1,0
256,W389724067GDGOAZ18E9,CA0389700,UC-SAN FRANCISCO PD,2024-03-04,San Francisco,SAN FRANCISCO,2,1,1,0,0,30,2,0.0000,0,0,0,0,0,0,0,0,1,0
283,U3800240793C646E114C,CA0380000,SAN FRANCISCO CO SO,2024-03-19,San Francisco,SAN FRANCISCO,2,1,1,0,0,40,2,0.0000,0,0,0,0,0,0,0,0,1,0
325,W389724063BZ8GXQ3479,CA0389700,UC-SAN FRANCISCO PD,2024-02-23,San Francisco,SAN FRANCISCO,2,1,1,0,0,30,1,0.0000,0,0,0,0,0,0,0,1,0,0
332,U3800240887D7A01BEDA,CA0380000,SAN FRANCISCO CO SO,2024-03-28,San Francisco,SAN FRANCISCO,7,1,1,0,0,50,2,0.0000,0,0,0,0,0,0,0,0,1,0


In [116]:
cali_force_df = cali_force_df.add_prefix("CALI_")


In [117]:
cali_force_cols = ["CALI_" + c for c in cali_force_cols]

In [118]:
cali_race_dict = {
    1: "Asian",
    2: "Black",
    3: "Hispanic",
    4: "Middle Eastern",
    5: "Native American",
    6: "Pacific Islander",
    7: "White",
    8: "Multiracial"
}

cali_gender_dict = {
    1: "Male",
    2: "Female",
    3: "Trans Male",
    4: "Trans Female",
    5: "Nonbinary"
}

In [119]:
cali_force_df = (
    cali_force_df.assign(
        date = lambda x: x["CALI_DATE_OF_STOP"],
        agency_name = lambda x: x["CALI_AGENCY_NAME"],
        ori = lambda x: x["CALI_AGENCY_ORI"],
        location = lambda x: x["CALI_county_name"],
        subject_race = lambda x: x["CALI_RAE_FULL"].map(cali_race_dict),
        subject_age = lambda x: x["CALI_AGE"],
        subject_gender = lambda x: x["CALI_G_FULL"].map(cali_gender_dict),
        force_used = lambda x: x[cali_force_cols].apply(
            lambda row: ' | '.join([col for col in cali_force_cols if row[col] == 1]), axis=1
        ),
        indication_of_injury = "",
        incident_id = lambda x: x[["CALI_DOJ_RECORD_ID"]].apply(
                lambda row: [str(v) for v in row if pd.notna(v)],
                axis=1
            ))
)
cali_force_df.head()

,CALI_DOJ_RECORD_ID,CALI_AGENCY_ORI,CALI_AGENCY_NAME,CALI_DATE_OF_STOP,CALI_county_name,CALI_LOC_CLOSEST_CITY,CALI_RAE_FULL,CALI_G_FULL,CALI_SOR_STRAIGHT,CALI_PD_FULL,CALI_LIMITED_ENGLISH_FLUENCY,CALI_AGE,CALI_REASON_FOR_STOP,CALI_OFA_NONE,CALI_OFA_BATON_USED,CALI_OFA_CHEM_SPRAY,CALI_OFA_ELECT_DEVICE_STUN,CALI_OFA_ELECT_DEVICE_DART,CALI_OFA_FIREARM_DISCHARGE,CALI_OFA_IMPACT_PROJECTILE_DISCHARGE,CALI_OFA_CANINE_BITE,CALI_OFA_REMOVED_VEHICLE_PHYCONTACT,CALI_OFA_PHYSICAL_COMPLIANCE,CALI_OFA_USE_VEHICLE,date,agency_name,ori,location,subject_race,subject_age,subject_gender,force_used,indication_of_injury,incident_id
51,U3800240116626931439,CA0380000,SAN FRANCISCO CO SO,2024-01-11,San Francisco,SAN FRANCISCO,2,1,1,0,0,35,4,0.0000,0,0,0,0,0,0,0,1,1,0,2024-01-11,SAN FRANCISCO CO SO,CA0380000,San Francisco,Black,35,Male,CALI_OFA_REMOVED_VEHICLE_PHYCONTACT | CALI_OFA...,,[U3800240116626931439]
256,W389724067GDGOAZ18E9,CA0389700,UC-SAN FRANCISCO PD,2024-03-04,San Francisco,SAN FRANCISCO,2,1,1,0,0,30,2,0.0000,0,0,0,0,0,0,0,0,1,0,2024-03-04,UC-SAN FRANCISCO PD,CA0389700,San Francisco,Black,30,Male,CALI_OFA_PHYSICAL_COMPLIANCE,,[W389724067GDGOAZ18E9]
283,U3800240793C646E114C,CA0380000,SAN FRANCISCO CO SO,2024-03-19,San Francisco,SAN FRANCISCO,2,1,1,0,0,40,2,0.0000,0,0,0,0,0,0,0,0,1,0,2024-03-19,SAN FRANCISCO CO SO,CA0380000,San Francisco,Black,40,Male,CALI_OFA_PHYSICAL_COMPLIANCE,,[U3800240793C646E114C]
325,W389724063BZ8GXQ3479,CA0389700,UC-SAN FRANCISCO PD,2024-02-23,San Francisco,SAN FRANCISCO,2,1,1,0,0,30,1,0.0000,0,0,0,0,0,0,0,1,0,0,2024-02-23,UC-SAN FRANCISCO PD,CA0389700,San Francisco,Black,30,Male,CALI_OFA_REMOVED_VEHICLE_PHYCONTACT,,[W389724063BZ8GXQ3479]
332,U3800240887D7A01BEDA,CA0380000,SAN FRANCISCO CO SO,2024-03-28,San Francisco,SAN FRANCISCO,7,1,1,0,0,50,2,0.0000,0,0,0,0,0,0,0,0,1,0,2024-03-28,SAN FRANCISCO CO SO,CA0380000,San Francisco,White,50,Male,CALI_OFA_PHYSICAL_COMPLIANCE,,[U3800240887D7A01BEDA]


In [120]:
cali_force_df.info()

<class 'pandas.DataFrame'>
Index: 36375 entries, 51 to 5065417
Data columns (total 34 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   CALI_DOJ_RECORD_ID                    36375 non-null  str    
 1   CALI_AGENCY_ORI                       36375 non-null  str    
 2   CALI_AGENCY_NAME                      36375 non-null  str    
 3   CALI_DATE_OF_STOP                     36375 non-null  str    
 4   CALI_county_name                      36375 non-null  str    
 5   CALI_LOC_CLOSEST_CITY                 36375 non-null  str    
 6   CALI_RAE_FULL                         36375 non-null  int8   
 7   CALI_G_FULL                           36375 non-null  int8   
 8   CALI_SOR_STRAIGHT                     36375 non-null  int8   
 9   CALI_PD_FULL                          36375 non-null  int8   
 10  CALI_LIMITED_ENGLISH_FLUENCY          36375 non-null  int8   
 11  CALI_AGE                    

## Concating

In [121]:
# Add a 'state' column to each dataframe before concatenation
nj_df_with_state = nj_df.copy()
nj_df_with_state["state"] = "NJ"

ohio_df_with_state = ohio_df.copy()
ohio_df_with_state["state"] = "OH"

ny_df_with_state = ny_df.copy()
ny_df_with_state["state"] = "NY"

ny_uof_long_with_state = ny_uof_long.copy()
ny_uof_long_with_state["state"] = "NY"

cali_force_df_with_state = cali_force_df.copy()
cali_force_df_with_state["state"] = "CA"

incidents_df = pd.concat([
    nj_df_with_state,
    ohio_df_with_state,
    ny_df_with_state,
    ny_uof_long_with_state,
    cali_force_df_with_state
])

incidents_df = incidents_df[[col for col in standard_cols if col in incidents_df.columns] +
                            [col for col in incidents_df.columns if col not in standard_cols]]
incidents_df.head()

,date,agency_name,ori,state,location,subject_race,subject_age,subject_gender,force_used,indication_of_injury,incident_id,NJ_Form ID,NJ_County,NJ_Agency Name,NJ_Incident ID,NJ_Report Number,NJ_Incident Case Number,NJ_Incident Date,NJ_Incident Type,NJ_Subject Actions,NJ_Subject Resistance,NJ_Subject Medical Treatment,NJ_Subject Injury Type,NJ_Subject Arrested,NJ_Reason Subject Not Arrested,NJ_Subject Type,NJ_Subject Age,NJ_Subject Race/Ethnicity,NJ_Subject Gender,NJ_Force Type,Ohio_AgencyName,Ohio_IncidentNumber,Ohio_IncidentDate,Ohio_InitialContactCircumstances,Ohio_LocationType,Ohio_SubjectResistanceTypes,Ohio_SubjectArmed/BelievedToBeArmed,Ohio_SubjectRaceEthnicity,Ohio_SubjectGender,Ohio_SubjectInjuries,Ohio_SubjectImpairments,Ohio_OfficerType,Ohio_OfficerResponse/ForceTypes,Ohio_OfficerRaceEthnicity,Ohio_OfficerGender,Ohio_OfficerInjuries,NY_TRI Incident Number,NY_ForceType,NY_Occurrence Date,NY_Incident Pct,NY_Patrol Borough,NY_YearMonthShort,NY_BasisForEncounter,NY_Subject Injury Level,NY_Subject Injured,NY_Age,NY_Subject Race,NY_Subject Gender,NY_Force Against MOS,NY_Subject Used Force,NY_incident_id,NY_Response ID\n(5/16/25 File),NY_Agency Name,NY_Incident Date,NY_City/town/village (location Where Incident Occurred),NY_County (location where incident occurred),NY_Circumstance,NY_Officer 1 - Use of Force Type,NY_Officer 1 Age,NY_Officer 1 Sex,NY_Officer 1 Race,NY_Officer 1 Ethnicity,NY_Officer 2 - Use of Force Type,NY_Officer 2 Age,NY_Officer 2 Sex,NY_Officer 2 Race,NY_Officer 2 Ethnicity,NY_Officer 3 - Use of Force Type,NY_Officer 3 Age,NY_Officer 3 Sex,NY_Officer 3 Race,NY_Officer 3 Ethnicity,NY_Officer 4 - Use of Force Type,NY_Officer 4 Age,NY_Officer 4 Sex,NY_Officer 4 Race,NY_Officer 4 Ethnicity,NY_Officer 5 - Use of Force Type,NY_Officer 5 Age,NY_Officer 5 Sex,NY_Officer 5 Race,NY_Officer 5 Ethnicity,NY_Officer 6 - Use of Force Type,NY_Officer 6 Age,NY_Officer 6 Sex,NY_Officer 6 Race,NY_Officer 6 Ethnicity,NY_Officer 7 - Use of Force Type,NY_Officer 7 Age,NY_Officer 7 Sex,NY_Officer 7 Race,NY_Officer 7 Ethnicity,NY_Officer 8 - Use of Force Type,NY_Officer 8 Age,NY_Officer 8 Sex,NY_Officer 8 Race,NY_Officer 8 Ethnicity,NY_Officer 9 - Use of Force Type,NY_Officer 9 Age,NY_Officer 9 Sex,NY_Officer 9 Race,NY_Officer 9 Ethnicity,NY_Officer 10 - Use of Force Type,NY_Officer 10 Age,NY_Officer 10 Sex,NY_Officer 10 Race,NY_Officer 10 Ethnicity,NY_Officer 11 - Use of Force Type,NY_Officer 11 Age,NY_Officer 11 Sex,NY_Officer 11 Race,NY_Officer 11 Ethnicity,NY_Officer 12 - Use of Force Type,NY_Officer 12 Age,NY_Officer 12 Sex,NY_Officer 12 Race,NY_Officer 12 Ethnicity,NY_Officer 13 - Use of Force Type,NY_Officer 13 Age,NY_Officer 13 Sex,NY_Officer 13 Race,NY_Officer 13 Ethnicity,NY_Officer 14 - Use of Force Type,NY_Officer 14 Age,NY_Officer 14 Sex,NY_Officer 14 Race,NY_Officer 14 Ethnicity,NY_Officer 15 - Use of Force Type,NY_Officer 15 Age,NY_Officer 15 Sex,NY_Officer 15 Race,NY_Officer 15 Ethnicity,NY_Officer 16 - Use of Force Type,NY_Officer 16 Age,NY_Officer 16 Sex,NY_Officer 16 Race,NY_Officer 16 Ethnicity,NY_subject_number,NY_subject_race,NY_subject_ethnicity,NY_subject_age,NY_subject_sex,CALI_DOJ_RECORD_ID,CALI_AGENCY_ORI,CALI_AGENCY_NAME,CALI_DATE_OF_STOP,CALI_county_name,CALI_LOC_CLOSEST_CITY,CALI_RAE_FULL,CALI_G_FULL,CALI_SOR_STRAIGHT,CALI_PD_FULL,CALI_LIMITED_ENGLISH_FLUENCY,CALI_AGE,CALI_REASON_FOR_STOP,CALI_OFA_NONE,CALI_OFA_BATON_USED,CALI_OFA_CHEM_SPRAY,CALI_OFA_ELECT_DEVICE_STUN,CALI_OFA_ELECT_DEVICE_DART,CALI_OFA_FIREARM_DISCHARGE,CALI_OFA_IMPACT_PROJECTILE_DISCHARGE,CALI_OFA_CANINE_BITE,CALI_OFA_REMOVED_VEHICLE_PHYCONTACT,CALI_OFA_PHYSICAL_COMPLIANCE,CALI_OFA_USE_VEHICLE
0,11/6/2023,Berkeley Twp PD,None,NJ,Ocean,Hispanic,28,Male,Used take down on,None,"[OCEAN-BERKELEY TWP PD-23-003647, UOF23-11-52,...",866515.0000,Ocean,Berkeley Twp PD,OCEAN-BERKELEY TWP PD-23-003647,UOF23-11-52,23-003647,11/6/2023,Assault,Attempt to escape from Custody,Passive Resistor,NaN,NaN,TRUE,NaN,Person,28,Hispanic,Male,Used take down 

In [122]:
incidents_path = "./data/incidents.csv"
incidents_df.to_csv(incidents_path, index=False)

# EDA 

In [123]:
incidents_df.info()

<class 'pandas.DataFrame'>
Index: 238976 entries, 0 to 5065417
Columns: 176 entries, date to CALI_OFA_USE_VEHICLE
dtypes: datetime64[us](1), float32(19), float64(23), object(6), str(127)
memory usage: 305.4+ MB


In [124]:
info_df = get_info(incidents_df[standard_cols].drop(columns=["incident_id"]))
info_df.style.background_gradient(cmap='viridis')

,Feature,No. of Missing Values,% of Missing Values,No. of Unique Values,DataType
0,date,0,0.000000,5639,object
1,agency_name,0,0.000000,1653,object
2,ori,202601,84.778806,461,object
3,state,0,0.000000,4,str
4,location,120,0.050214,1611,str
5,subject_race,1,0.000418,31,str
6,subject_age,6908,2.890667,181,object
7,subject_gender,271,0.113401,12,str
8,force_used,0,0.000000,2040,str
9,indication_of_injury,143041,59.855801,31,object


In [125]:
incidents_df.shape

(238976, 176)

In [126]:
incidents_df[incidents_df["subject_gender"].str.contains(",", regex=False)]

,date,agency_name,ori,state,location,subject_race,subject_age,subject_gender,force_used,indication_of_injury,incident_id,NJ_Form ID,NJ_County,NJ_Agency Name,NJ_Incident ID,NJ_Report Number,NJ_Incident Case Number,NJ_Incident Date,NJ_Incident Type,NJ_Subject Actions,NJ_Subject Resistance,NJ_Subject Medical Treatment,NJ_Subject Injury Type,NJ_Subject Arrested,NJ_Reason Subject Not Arrested,NJ_Subject Type,NJ_Subject Age,NJ_Subject Race/Ethnicity,NJ_Subject Gender,NJ_Force Type,Ohio_AgencyName,Ohio_IncidentNumber,Ohio_IncidentDate,Ohio_InitialContactCircumstances,Ohio_LocationType,Ohio_SubjectResistanceTypes,Ohio_SubjectArmed/BelievedToBeArmed,Ohio_SubjectRaceEthnicity,Ohio_SubjectGender,Ohio_SubjectInjuries,Ohio_SubjectImpairments,Ohio_OfficerType,Ohio_OfficerResponse/ForceTypes,Ohio_OfficerRaceEthnicity,Ohio_OfficerGender,Ohio_OfficerInjuries,NY_TRI Incident Number,NY_ForceType,NY_Occurrence Date,NY_Incident Pct,NY_Patrol Borough,NY_YearMonthShort,NY_BasisForEncounter,NY_Subject Injury Level,NY_Subject Injured,NY_Age,NY_Subject Race,NY_Subject Gender,NY_Force Against MOS,NY_Subject Used Force,NY_incident_id,NY_Response ID\n(5/16/25 File),NY_Agency Name,NY_Incident Date,NY_City/town/village (location Where Incident Occurred),NY_County (location where incident occurred),NY_Circumstance,NY_Officer 1 - Use of Force Type,NY_Officer 1 Age,NY_Officer 1 Sex,NY_Officer 1 Race,NY_Officer 1 Ethnicity,NY_Officer 2 - Use of Force Type,NY_Officer 2 Age,NY_Officer 2 Sex,NY_Officer 2 Race,NY_Officer 2 Ethnicity,NY_Officer 3 - Use of Force Type,NY_Officer 3 Age,NY_Officer 3 Sex,NY_Officer 3 Race,NY_Officer 3 Ethnicity,NY_Officer 4 - Use of Force Type,NY_Officer 4 Age,NY_Officer 4 Sex,NY_Officer 4 Race,NY_Officer 4 Ethnicity,NY_Officer 5 - Use of Force Type,NY_Officer 5 Age,NY_Officer 5 Sex,NY_Officer 5 Race,NY_Officer 5 Ethnicity,NY_Officer 6 - Use of Force Type,NY_Officer 6 Age,NY_Officer 6 Sex,NY_Officer 6 Race,NY_Officer 6 Ethnicity,NY_Officer 7 - Use of Force Type,NY_Officer 7 Age,NY_Officer 7 Sex,NY_Officer 7 Race,NY_Officer 7 Ethnicity,NY_Officer 8 - Use of Force Type,NY_Officer 8 Age,NY_Officer 8 Sex,NY_Officer 8 Race,NY_Officer 8 Ethnicity,NY_Officer 9 - Use of Force Type,NY_Officer 9 Age,NY_Officer 9 Sex,NY_Officer 9 Race,NY_Officer 9 Ethnicity,NY_Officer 10 - Use of Force Type,NY_Officer 10 Age,NY_Officer 10 Sex,NY_Officer 10 Race,NY_Officer 10 Ethnicity,NY_Officer 11 - Use of Force Type,NY_Officer 11 Age,NY_Officer 11 Sex,NY_Officer 11 Race,NY_Officer 11 Ethnicity,NY_Officer 12 - Use of Force Type,NY_Officer 12 Age,NY_Officer 12 Sex,NY_Officer 12 Race,NY_Officer 12 Ethnicity,NY_Officer 13 - Use of Force Type,NY_Officer 13 Age,NY_Officer 13 Sex,NY_Officer 13 Race,NY_Officer 13 Ethnicity,NY_Officer 14 - Use of Force Type,NY_Officer 14 Age,NY_Officer 14 Sex,NY_Officer 14 Race,NY_Officer 14 Ethnicity,NY_Officer 15 - Use of Force Type,NY_Officer 15 Age,NY_Officer 15 Sex,NY_Officer 15 Race,NY_Officer 15 Ethnicity,NY_Officer 16 - Use of Force Type,NY_Officer 16 Age,NY_Officer 16 Sex,NY_Officer 16 Race,NY_Officer 16 Ethnicity,NY_subject_number,NY_subject_race,NY_subject_ethnicity,NY_subject_age,NY_subject_sex,CALI_DOJ_RECORD_ID,CALI_AGENCY_ORI,CALI_AGENCY_NAME,CALI_DATE_OF_STOP,CALI_county_name,CALI_LOC_CLOSEST_CITY,CALI_RAE_FULL,CALI_G_FULL,CALI_SOR_STRAIGHT,CALI_PD_FULL,CALI_LIMITED_ENGLISH_FLUENCY,CALI_AGE,CALI_REASON_FOR_STOP,CALI_OFA_NONE,CALI_OFA_BATON_USED,CALI_OFA_CHEM_SPRAY,CALI_OFA_ELECT_DEVICE_STUN,CALI_OFA_ELECT_DEVICE_DART,CALI_OFA_FIREARM_DISCHARGE,CALI_OFA_IMPACT_PROJECTILE_DISCHARGE,CALI_OFA_CANINE_BITE,CALI_OFA_REMOVED_VEHICLE_PHYCONTACT,CALI_OFA_PHYSICAL_COMPLIANCE,CALI_OFA_USE_VEHICLE


In [127]:
incidents_df["state"].value_counts()

state
NY    101466
NJ     97877
CA     36375
OH      3258
Name: count, dtype: int64

In [128]:
incidents_df[lambda x: x.state=="NY"]["force_used"].value_counts()

force_used
Physical Force                                                                                                                                                               48068
Electrical Weapon                                                                                                                                                             8898
Pointed Firearm                                                                                                                                                               7717
Used/Deployed Electronic Control Weapon                                                                                                                                       7571
Used/Deployed Chemical Agent                                                                                                                                                  5027
                                                                                              